In [ ]:
"""
CAMUS Offset-SRVF Repeated Train/Test Pipeline with Training-Set CV
========================================================

Author
------
Bahadir Yuzbasi

Purpose
-------
This script implements a reproducible, GitHub-ready real-data analysis pipeline
for CAMUS-style cardiac ultrasound segmentations using an Offset-SRVF framework.

Main pipeline
-------------
1. Load CAMUS segmentation masks from database_nifti/
2. Extract endocardial and epicardial contours
3. Resample closed contours to a common grid
4. Estimate myocardial thickness r(s)
5. Build embedded curves

       X(s) = ( beta_in(s), alpha * r(s) )

6. Compute SRVF representations
7. Repeat train/test splitting
8. For each repetition:
      a. Select lambda via K-fold CV on training set
      b. Fit alignment template on training data
      c. Align test data to training template
      d. Extract features
      e. Tune ML models via cross-validation on training set
      f. Evaluate final models on test set only

Important methodological rule
-----------------------------
All tuning (lambda and ML hyperparameters) is performed
using training data only via cross-validation.
The test set is used exclusively for final evaluation.

Expected directory structure
----------------------------
Input:
    database_nifti/

Output:
    real_data_analysis/
        figures/
        tables/
        data/
        tex/
"""

# ============================================================
# 0. Imports
# ============================================================

import os
import re
import glob
import json
import warnings
import itertools
from pathlib import Path

import numpy as np
import sys
NUMERICS_DIR = Path("methods").resolve()
if str(NUMERICS_DIR) not in sys.path:
    sys.path.insert(0, str(NUMERICS_DIR))
import srvf_numerics as _srvf_numerics
import camus_inner_cv as _camus_cv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import cv2
import nibabel as nib

from matplotlib.gridspec import GridSpec
from matplotlib.collections import LineCollection
from scipy.interpolate import interp1d
from scipy.ndimage import distance_transform_edt, gaussian_filter1d, map_coordinates

from sklearn.model_selection import train_test_split, ParameterGrid, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.stats import ttest_ind

from sklearn.manifold import TSNE
from scipy.stats import ttest_ind, mannwhitneyu

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    LogisticRegression
)

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.svm import SVR, SVC

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False


# ============================================================
# 1. Configuration
# ============================================================

ROOT_DIR = Path("database_nifti")

VIEW_KEYWORD = "2CH"
PHASE_KEYWORD = "ED"

N = 250
s = np.linspace(0.0, 1.0, N, endpoint=False)
dt = 1.0 / N

ALPHA_EMBED = 2.0

MAX_PATIENTS = 500
ENDO_LABEL = 1
MYO_LABEL = 2
MIN_PIXELS = 20
THICKNESS_SMOOTH_SIGMA = 2.0

N_ITER_ALIGN = 4
USE_CYCLIC_PREALIGNMENT = True

LAMBDA_GRID = [
    0.0,
    0.002, 0.005,
    0.01, 0.02, 0.05,
    0.1, 0.2, 0.5,
    0.5,
    1.0, 2.0, 4.0,
    8,16
]

N_REPEATS = 1
N_FOLDS = 5
TRAIN_SIZE = 0.70
TEST_SIZE = 0.30
RANDOM_STATE = 2026

N_PCA_COMPONENTS = 10
EF_CLASSIFICATION_THRESHOLD = 50.0

REAL_OUT = Path("runs/camus")
FIG_DIR = REAL_OUT / "figures"
TAB_DIR = REAL_OUT / "tables"
DATA_DIR = REAL_OUT / "data"
TEX_DIR = REAL_OUT / "tex"

# Each complete run requires a new, empty destination.
REAL_OUT.mkdir(parents=True, exist_ok=False)
for d in [FIG_DIR, TAB_DIR, DATA_DIR, TEX_DIR]:
    d.mkdir(parents=False, exist_ok=False)

mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["pdf.fonttype"] = 42


# ============================================================
# 2. Saving utilities
# ============================================================

def save_figure(fig, name, caption=None, label=None):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.eps", format="eps", bbox_inches="tight")

    if caption is not None and label is not None:
        tex = rf"""
\begin{{figure}}[!ht]
\centering
\includegraphics[width=0.95\textwidth]{{figures/{name}.pdf}}
\caption{{{caption}}}
\label{{{label}}}
\end{{figure}}
"""
        with open(TEX_DIR / f"{name}.tex", "w") as f:
            f.write(tex.strip() + "\n")


def save_latex_table(df, name, caption, label):
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)

    tex = df.to_latex(
        index=False,
        float_format="%.6f",
        caption=caption,
        label=label
    )

    with open(TEX_DIR / f"{name}.tex", "w") as f:
        f.write(tex)


def save_json(obj, name):
    with open(DATA_DIR / f"{name}.json", "w") as f:
        json.dump(obj, f, indent=4)


# ============================================================
# 3. Geometry utilities
# ============================================================

def close_for_plot(curve):
    return np.vstack([curve, curve[0]])


def periodic_derivative(curve):
    return (np.roll(curve, -1, axis=0) - np.roll(curve, 1, axis=0)) / (2.0 * dt)


def signed_area(curve):
    x = curve[:, 0]
    y = curve[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - y * np.roll(x, -1))


def ensure_ccw(curve):
    c = curve.copy()
    if signed_area(c) < 0:
        c = c[::-1]
    return c


def reindex_by_rightmost(curve):
    c = curve.copy()
    centroid = c.mean(axis=0)
    idx = np.argmax(c[:, 0] - centroid[0])
    return np.vstack([c[idx:], c[:idx]])


def resample_closed_curve(curve, n_points=N):
    c = np.asarray(curve, dtype=float)

    if len(c) < 5:
        return None

    if not np.allclose(c[0], c[-1]):
        c = np.vstack([c, c[0]])

    d = np.sqrt(np.sum(np.diff(c, axis=0) ** 2, axis=1))
    arclen = np.insert(np.cumsum(d), 0, 0.0)

    if arclen[-1] < 1e-8:
        return None

    arclen = arclen / arclen[-1]

    fx = interp1d(arclen, c[:, 0], kind="linear")
    fy = interp1d(arclen, c[:, 1], kind="linear")

    grid = np.linspace(0.0, 1.0, n_points, endpoint=False)
    out = np.column_stack([fx(grid), fy(grid)])

    out = ensure_ccw(out)
    out = reindex_by_rightmost(out)

    return out


def unit_tangent_normal(beta):
    db = periodic_derivative(beta)
    speed = np.linalg.norm(db, axis=1, keepdims=True)
    speed = np.maximum(speed, 1e-10)

    T = db / speed
    Nvec = np.column_stack([T[:, 1], -T[:, 0]])

    centroid = beta.mean(axis=0, keepdims=True)
    radial = beta - centroid

    if np.mean(np.sum(Nvec * radial, axis=1)) < 0:
        Nvec = -Nvec

    return T, Nvec


def reconstruct_outer(beta_in, r):
    _, Nvec = unit_tangent_normal(beta_in)
    return beta_in + r[:, None] * Nvec


def build_embedding(beta_in, r):
    return np.column_stack([beta_in, ALPHA_EMBED * r])


def curve_area(beta):
    return abs(signed_area(beta))


def curve_perimeter(beta):
    B = close_for_plot(beta)
    return np.sum(np.sqrt(np.sum(np.diff(B, axis=0) ** 2, axis=1)))


def compactness(beta):
    perim = curve_perimeter(beta)
    if perim < 1e-10:
        return np.nan
    return 4.0 * np.pi * curve_area(beta) / (perim ** 2)

def inverse_srvf(q, x0=None):
    if x0 is None:
        x0 = np.zeros(q.shape[1])

    vel = q * np.linalg.norm(q, axis=1, keepdims=True)

    X = np.zeros_like(q)
    X[0] = x0

    for j in range(1, len(q)):
        X[j] = X[j - 1] + 0.5 * (vel[j - 1] + vel[j]) * dt

    return X


def enforce_curve_closure(X):
    """
    Enforce X(1) = X(0) for a closed curve in R^d.
    Minimal correction: shift entire curve so last point matches first.
    """
    Xc = X.copy()
    delta = Xc[-1] - Xc[0]
    Xc = Xc - np.linspace(0, 1, len(Xc))[:, None] * delta
    Xc[-1] = Xc[0]
    return Xc
    
# ============================================================
# 4. Medical image loading
# ============================================================

def find_candidate_mask_files(patient_dir):
    patterns = ["*.nii", "*.nii.gz", "*.mhd", "*.mha"]
    files = []

    for p in patterns:
        files.extend(glob.glob(str(patient_dir / "**" / p), recursive=True))

    return files


def choose_mask_file(patient_dir):
    candidates = find_candidate_mask_files(patient_dir)
    scored = []

    for f in candidates:
        name = os.path.basename(f).lower()

        score = 0
        if VIEW_KEYWORD.lower() in name:
            score += 3
        if PHASE_KEYWORD.lower() in name:
            score += 3
        if any(k in name for k in ["gt", "seg", "label", "mask"]):
            score += 2
        if "image" in name:
            score -= 3

        scored.append((score, f))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    if len(scored) == 0 or scored[0][0] <= 0:
        return None

    return scored[0][1]

def choose_image_file(patient_dir):
    candidates = find_candidate_mask_files(patient_dir)
    scored = []

    for f in candidates:
        name = os.path.basename(f).lower()

        score = 0
        if VIEW_KEYWORD.lower() in name:
            score += 3
        if PHASE_KEYWORD.lower() in name:
            score += 3
        if any(k in name for k in ["gt", "seg", "label", "mask"]):
            score -= 5
        if "image" in name:
            score += 3

        scored.append((score, f))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    if len(scored) == 0 or scored[0][0] <= 0:
        return None

    return scored[0][1]


def load_medical_image(path):
    path = str(path)

    if path.lower().endswith(".nii") or path.lower().endswith(".nii.gz"):
        arr = nib.load(path).get_fdata()
    else:
        import SimpleITK as sitk
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = np.transpose(arr, (1, 2, 0))

    arr = np.asarray(arr)

    if arr.ndim == 2:
        img2d = arr
    elif arr.ndim == 3:
        sums = [np.sum(np.abs(arr[..., k])) for k in range(arr.shape[-1])]
        idx = int(np.argmax(sums))
        img2d = arr[..., idx]
    else:
        raise ValueError("Unsupported image dimension.")

    img2d = img2d.astype(float)
    img2d = (img2d - np.nanmin(img2d)) / (np.nanmax(img2d) - np.nanmin(img2d) + 1e-12)

    return img2d
    
def load_medical_mask(path):
    path = str(path)

    if path.lower().endswith(".nii") or path.lower().endswith(".nii.gz"):
        arr = nib.load(path).get_fdata()
    else:
        import SimpleITK as sitk
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = np.transpose(arr, (1, 2, 0))

    arr = np.asarray(arr)

    if arr.ndim == 2:
        mask2d = arr
    elif arr.ndim == 3:
        sums = [np.sum(arr[..., k] > 0) for k in range(arr.shape[-1])]
        idx = int(np.argmax(sums))
        mask2d = arr[..., idx]
    else:
        raise ValueError(f"Unsupported image dimension: {path}")

    return np.round(mask2d).astype(np.int32)


def get_largest_contour(binary_mask):
    binary_mask = binary_mask.astype(np.uint8)

    contours, _ = cv2.findContours(
        binary_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours) == 0:
        return None

    cnt = max(contours, key=lambda x: len(x))
    return cnt[:, 0, :].astype(float)


def extract_contours(mask):
    endo_bin = (mask == ENDO_LABEL).astype(np.uint8)
    myo_bin = (mask == MYO_LABEL).astype(np.uint8)

    if endo_bin.sum() < MIN_PIXELS:
        return None, None, None

    epi_region = ((endo_bin > 0) | (myo_bin > 0)).astype(np.uint8)

    if epi_region.sum() < MIN_PIXELS:
        return None, None, None

    inner = get_largest_contour(endo_bin)
    outer = get_largest_contour(epi_region)

    return inner, outer, epi_region


def thickness_from_distance_map(beta_in, epi_region):
    dist_map = distance_transform_edt(epi_region.astype(np.uint8))

    rows = beta_in[:, 1]
    cols = beta_in[:, 0]

    r = map_coordinates(
        dist_map.astype(float),
        [rows, cols],
        order=1,
        mode="nearest"
    )

    r = gaussian_filter1d(r, sigma=THICKNESS_SMOOTH_SIGMA)
    r = np.maximum(r, 0.0)

    return r


# ============================================================
# 5. Optional clinical metadata
# ============================================================

def parse_numeric_value(text, keys):
    for key in keys:
        pattern = rf"{key}\s*[:=]\s*([-+]?\d*\.?\d+)"
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return np.nan


def parse_patient_metadata(patient_dir):
    files = []
    for ext in ["*.cfg", "*.txt", "*.csv"]:
        files.extend(glob.glob(str(patient_dir / "**" / ext), recursive=True))

    text = ""

    for f in files:
        try:
            with open(f, "r", errors="ignore") as fh:
                text += "\n" + fh.read()
        except Exception:
            pass

    return {
        "EF": parse_numeric_value(text, ["EF", "Ejection Fraction"]),
        "EDV": parse_numeric_value(text, ["EDV", "End Diastolic Volume"]),
        "ESV": parse_numeric_value(text, ["ESV", "End Systolic Volume"])
    }


# ============================================================
# 5.1 Clustering utilities
# ============================================================

def clustering_model_specs():
    return {
        "kmeans": {
            "fit": lambda X, k: KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X)
        },
        "gmm": {
            "fit": lambda X, k: GaussianMixture(n_components=k, random_state=RANDOM_STATE).fit(X)
        }
    }

def select_k_via_validation(X_train, X_val):
    results = []
    best_k = None
    best_score = -np.inf
    best_model = None
    best_name = None

    for model_name, spec in clustering_model_specs().items():
        for k in [2, 3, 4, 5, 6]:

            if model_name == "hierarchical":
                labels_val = spec["fit"](X_val, k).labels_
            else:
                model = spec["fit"](X_train, k)
                if hasattr(model, "predict"):
                    labels_val = model.predict(X_val)
                else:
                    labels_val = model.fit_predict(X_val)

            if len(np.unique(labels_val)) < 2:
                continue

            score = silhouette_score(X_val, labels_val)

            results.append({
                "model": model_name,
                "k": k,
                "silhouette": score
            })

            if score > best_score:
                best_score = score
                best_k = k
                best_name = model_name
                best_model = spec["fit"](X_train, k)

    return best_model, best_name, best_k, pd.DataFrame(results)


def assign_clusters(model, model_name, X):
    if model_name == "hierarchical":
        return model.fit_predict(X)

    if hasattr(model, "predict"):
        return model.predict(X)

    return model.fit_predict(X)


def cluster_summary(metadata, labels):
    df = metadata.copy()
    df["cluster"] = labels

    available = [
        c for c in ["EF", "EDV", "ESV"]
        if c in df.columns
    ]

    if len(available) == 0:
        return df.groupby("cluster").size().reset_index(name="n")

    summary = df.groupby("cluster")[available].agg(["mean", "std", "count"])
    summary.columns = ["_".join(col) for col in summary.columns]
    summary = summary.reset_index()

    return summary

def normal_ef_cluster_analysis(metadata, features, beta, r, labels):
    df = metadata.copy()
    df["cluster"] = labels

    normal_idx = np.where(df["EF"].values >= EF_CLASSIFICATION_THRESHOLD)[0]

    if len(normal_idx) < 6:
        return pd.DataFrame(), pd.DataFrame(), None

    df_normal = df.iloc[normal_idx].reset_index(drop=True)
    features_normal = features.iloc[normal_idx].reset_index(drop=True)
    beta_normal = beta[normal_idx]
    r_normal = r[normal_idx]
    labels_normal = labels[normal_idx]

    if len(np.unique(labels_normal)) < 2:
        return pd.DataFrame(), pd.DataFrame(), None

    # Add structural features to metadata
    df_normal["area"] = features_normal["area"].values
    df_normal["perimeter"] = features_normal["perimeter"].values
    df_normal["compactness"] = features_normal["compactness"].values
    df_normal["mean_thickness"] = features_normal["mean_thickness"].values
    df_normal["sd_thickness"] = features_normal["sd_thickness"].values
    df_normal["shape_distance_to_template"] = features_normal["shape_distance_to_template"].values
    df_normal["srvf_distance_to_template"] = features_normal["srvf_distance_to_template"].values
    df_normal["cluster"] = labels_normal

    metrics = [
        "EF",
        "EDV",
        "ESV",
        "area",
        "perimeter",
        "compactness",
        "mean_thickness",
        "sd_thickness",
        "shape_distance_to_template",
        "srvf_distance_to_template"
    ]

    summary = (
        df_normal
        .groupby("cluster")[metrics]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    summary.columns = [
        "_".join(col).strip("_")
        for col in summary.columns.values
    ]

    comparison_rows = []

    clusters = sorted(df_normal["cluster"].unique())

    if len(clusters) >= 2:
        c1, c2 = clusters[:2]

        for m in metrics:
            g1 = df_normal[df_normal["cluster"] == c1][m].dropna()
            g2 = df_normal[df_normal["cluster"] == c2][m].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            try:
                _, p_t = ttest_ind(g1, g2, equal_var=False)
            except Exception:
                p_t = np.nan

            try:
                _, p_mw = mannwhitneyu(g1, g2, alternative="two-sided")
            except Exception:
                p_mw = np.nan

            pooled_sd = np.sqrt(
                ((len(g1) - 1) * np.var(g1, ddof=1)
                 + (len(g2) - 1) * np.var(g2, ddof=1))
                / max(len(g1) + len(g2) - 2, 1)
            )

            cohen_d = (
                (np.mean(g1) - np.mean(g2)) / pooled_sd
                if pooled_sd > 1e-12 else np.nan
            )

            comparison_rows.append({
                "metric": m,
                "cluster_1": c1,
                "cluster_2": c2,
                "cluster_1_mean": np.mean(g1),
                "cluster_2_mean": np.mean(g2),
                "mean_difference": np.mean(g1) - np.mean(g2),
                "cohen_d": cohen_d,
                "p_ttest": p_t,
                "p_mannwhitney": p_mw
            })

    comparison = pd.DataFrame(comparison_rows)

    normal_payload = {
        "metadata": df_normal,
        "features": features_normal,
        "beta": beta_normal,
        "r": r_normal,
        "labels": labels_normal
    }

    return summary, comparison, normal_payload


def enforce_scalar_periodicity(r):
    r = r.copy()
    r[-1] = r[0]
    return r


def karcher_template_from_cluster(beta_cluster, r_cluster):
    X_cluster = np.array([
        build_embedding(beta_cluster[i], r_cluster[i])
        for i in range(len(beta_cluster))
    ])

    q_cluster = np.array([
        normalize_srvf(srvf(X_cluster[i]))
        for i in range(len(X_cluster))
    ])

    q_mean = karcher_mean_sphere(
        q_cluster,
        max_iter=100,
        tol=1e-6,
        verbose=False
    )

    X_mean = inverse_srvf(q_mean, x0=np.zeros(3))
    X_mean = enforce_curve_closure(X_mean)

    # --------------------------------------------------------
    # Restore XY scale to cluster average perimeter
    # --------------------------------------------------------
    target_xy_length = np.mean([
        curve_perimeter(beta_cluster[i])
        for i in range(len(beta_cluster))
    ])

    current_xy_length = curve_perimeter(X_mean[:, :2])
    xy_scale = target_xy_length / (current_xy_length + 1e-8)

    xy_cent = X_mean[:, :2].mean(axis=0, keepdims=True)
    X_mean[:, :2] = xy_cent + xy_scale * (X_mean[:, :2] - xy_cent)

    # --------------------------------------------------------
    # Restore XY location to cluster average centroid
    # --------------------------------------------------------
    target_cent = beta_cluster.reshape(-1, 2).mean(axis=0, keepdims=True)
    current_cent = X_mean[:, :2].mean(axis=0, keepdims=True)
    X_mean[:, :2] += target_cent - current_cent

    # --------------------------------------------------------
    # Restore thickness profile from cluster aligned thickness
    # This prevents flat or collapsed mean thickness.
    # --------------------------------------------------------
    X_mean[:, 2] = ALPHA_EMBED * np.mean(r_cluster, axis=0)
    X_mean[:, 2] = enforce_scalar_periodicity(X_mean[:, 2])

    X_mean = enforce_curve_closure(X_mean)

    beta_mean = enforce_curve_closure(X_mean[:, :2])
    r_mean = enforce_scalar_periodicity(X_mean[:, 2] / ALPHA_EMBED)

    return beta_mean, r_mean, X_mean

    
def plot_normal_ef_cluster_shape_thickness(beta, r, labels, name):
    """
    Normal-EF structural cluster visualization:
    left  = cluster-wise mean shapes
    right = cluster-wise mean thickness profiles
    """

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    ax_shape = axes[0]
    ax_thick = axes[1]

    colors = [
        "#1f77b4",
        "#d62728",
        "#2ca02c",
        "#9467bd",
        "#ff7f0e",
        "#8c564b"
    ]

    for cc, c in enumerate(sorted(np.unique(labels))):

        idx = np.where(labels == c)[0]

        if len(idx) < 2:
            continue

        color = colors[cc % len(colors)]

        # ----------------------------------------------------
        # Cluster Karcher mean template
        # ----------------------------------------------------
        beta_mean, r_mean, X_mean = karcher_template_from_cluster(
            beta[idx],
            r[idx]
        )

        # Safety fallback if reconstructed shape collapses
        avg_cluster_perim = np.mean([
            curve_perimeter(beta[j])
            for j in idx
        ])

        if curve_perimeter(beta_mean) < 0.25 * avg_cluster_perim:
            beta_mean = enforce_curve_closure(
                np.mean(beta[idx], axis=0)
            )
            r_mean = enforce_scalar_periodicity(
                np.mean(r[idx], axis=0)
            )

        beta_out_mean = reconstruct_outer(beta_mean, r_mean)

        # ----------------------------------------------------
        # Background individual contours and thickness profiles
        # ----------------------------------------------------
        for j in idx:

            B = close_for_plot(beta[j])

            ax_shape.plot(
                B[:, 0],
                B[:, 1],
                color=color,
                alpha=0.10,
                lw=0.8
            )

            ax_thick.plot(
                s,
                r[j],
                color=color,
                alpha=0.10,
                lw=0.8
            )

        # ----------------------------------------------------
        # Mean shape
        # ----------------------------------------------------
        B_in = close_for_plot(beta_mean)
        B_out = close_for_plot(beta_out_mean)

        ax_shape.plot(
            B_in[:, 0],
            B_in[:, 1],
            color=color,
            lw=3.2,
            label=f"Cluster {c} endocardium"
        )

        ax_shape.plot(
            B_out[:, 0],
            B_out[:, 1],
            color=color,
            linestyle="--",
            lw=2.8,
            label=f"Cluster {c} epicardium"
        )

        # ----------------------------------------------------
        # Mean thickness with ±1 SD band
        # ----------------------------------------------------
        r_sd = np.std(r[idx], axis=0)

        ax_thick.fill_between(
            s,
            r_mean - r_sd,
            r_mean + r_sd,
            color=color,
            alpha=0.14
        )

        ax_thick.plot(
            s,
            r_mean,
            color=color,
            lw=3.0,
            label=f"Cluster {c} mean"
        )

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------
    ax_shape.set_title("Normal-EF structural clusters: ventricular shape")
    ax_shape.set_aspect("equal")
    ax_shape.set_xlabel(r"$\beta_x(s)$")
    ax_shape.set_ylabel(r"$\beta_y(s)$")
    ax_shape.grid(True, linestyle="--", alpha=0.20)
    ax_shape.legend(fontsize=8, loc="best")

    ax_thick.set_title("Normal-EF structural clusters: myocardial thickness")
    ax_thick.set_xlabel(r"Normalized contour parameter $s$")
    ax_thick.set_ylabel("Thickness")
    ax_thick.grid(True, linestyle="--", alpha=0.20)
    ax_thick.legend(fontsize=8, loc="best")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Structural heterogeneity among patients with normal EF. "
            "Left: cluster-wise mean ventricular shapes with endocardial and "
            "epicardial contours. Right: cluster-wise myocardial thickness "
            "profiles with pointwise plus/minus one standard deviation bands."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def plot_cluster_means(beta, r, labels, name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for c in np.unique(labels):
        idx = np.where(labels == c)[0]

        beta_mean, r_mean, X_mean = karcher_template_from_cluster(
            beta[idx],
            r[idx]
        )

        beta_out = reconstruct_outer(beta_mean, r_mean)

        B = close_for_plot(beta_mean)
        Bout = close_for_plot(beta_out)

        axes[0].plot(B[:,0], B[:,1], lw=3, label=f"C{c}")
        axes[0].plot(Bout[:,0], Bout[:,1], "--", lw=2)

        axes[1].plot(s, r_mean, lw=3, label=f"C{c}")

    axes[0].set_title("Mean shapes")
    axes[0].axis("equal")
    axes[0].legend()

    axes[1].set_title("Mean thickness")
    axes[1].legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Cluster-wise mean shapes and thickness profiles.",
        label=f"fig:{name}"
    )

    plt.close(fig)
    
# ============================================================
# 6. Build dataset
# ============================================================

def build_real_dataset():
    patient_dirs = sorted([
        ROOT_DIR / d
        for d in os.listdir(ROOT_DIR)
        if (ROOT_DIR / d).is_dir() and d.lower().startswith("patient")
    ])

    if MAX_PATIENTS is not None:
        patient_dirs = patient_dirs[:MAX_PATIENTS]

    beta_in_list = []
    beta_out_list = []
    r_list = []
    X_list = []
    rows = []

    for pdir in patient_dirs:
        patient_id = pdir.name
        f = choose_mask_file(pdir)

        if f is None:
            continue

        try:
            mask = load_medical_mask(f)

            inner_raw, outer_raw, epi_region = extract_contours(mask)

            if inner_raw is None or outer_raw is None:
                continue

            beta_in = resample_closed_curve(inner_raw, N)
            beta_out = resample_closed_curve(outer_raw, N)

            if beta_in is None or beta_out is None:
                continue

            r = thickness_from_distance_map(beta_in, epi_region)
            X = build_embedding(beta_in, r)

            beta_in_list.append(beta_in)
            beta_out_list.append(beta_out)
            r_list.append(r)
            X_list.append(X)

            meta = parse_patient_metadata(pdir)

            image_file = choose_image_file(pdir)

            meta.update({
                "patient_id": patient_id,
                "file_used": f,
                "image_file": image_file
            })

            rows.append(meta)

        except Exception as e:
            print(f"[WARN] Skipping {patient_id}: {e}")

    beta_in_raw = np.asarray(beta_in_list)
    beta_out_raw = np.asarray(beta_out_list)
    r_raw = np.asarray(r_list)
    X_raw = np.asarray(X_list)
    metadata = pd.DataFrame(rows)

    if len(beta_in_raw) == 0:
        raise RuntimeError("No valid CAMUS samples found.")

    return beta_in_raw, beta_out_raw, r_raw, X_raw, metadata


# ============================================================
# 7. SRVF utilities
# ============================================================

def srvf(X):
    dX = periodic_derivative(X)
    speed = np.linalg.norm(dX, axis=1)
    speed = np.maximum(speed, 1e-10)
    return dX / np.sqrt(speed)[:, None]


def l2_inner(q1, q2):
    return np.sum(np.sum(q1 * q2, axis=1)) * dt


def l2_norm(q):
    return np.sqrt(max(l2_inner(q, q), 0.0))


def normalize_srvf(q):
    nrm = l2_norm(q)
    if nrm < 1e-10:
        return q.copy()
    return q / nrm


def compute_all_srvfs(X_all):
    q_raw = np.array([srvf(X_all[i]) for i in range(len(X_all))])
    q_hat = np.array([normalize_srvf(q_raw[i]) for i in range(len(X_all))])
    return q_raw, q_hat


# ============================================================
# 8. Alignment utilities
# ============================================================

def interp_periodic(Y, gamma):
    """Warp endpoint-exclusive periodic data by an endpoint-inclusive map.

    DP gamma values are sampled at linspace(0, 1, len(gamma)), including
    both boundaries. Evaluate that map on the unique data grid first.
    """
    Y = np.asarray(Y)
    gamma = np.clip(np.asarray(gamma), 0.0, 1.0)
    grid = np.linspace(0.0, 1.0, len(Y), endpoint=False)
    grid_ext = np.r_[grid, 1.0]
    gamma_grid = np.linspace(0.0, 1.0, len(gamma))
    gamma_at_data = np.interp(grid, gamma_grid, gamma)

    if Y.ndim == 1:
        return np.interp(gamma_at_data, grid_ext, np.r_[Y, Y[0]])

    out = np.zeros_like(Y)
    for j in range(Y.shape[1]):
        out[:, j] = np.interp(
            gamma_at_data, grid_ext, np.r_[Y[:, j], Y[0, j]]
        )
    return out


def best_cyclic_shift(X_ref, X):
    best_shift = 0
    best_cost = np.inf

    for sh in range(N):
        Xs = np.roll(X, sh, axis=0)
        cost = np.mean(np.sum((X_ref - Xs) ** 2, axis=1))

        if cost < best_cost:
            best_cost = cost
            best_shift = sh

    return best_shift


def optimal_rotation_2d(A, B):
    M = A.T @ B
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt

    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R

def apply_gamma_to_srvf(q, gamma):
    """Apply q(gamma(s))*sqrt(gamma_dot(s)) on the unique periodic grid.

    gamma has endpoint-inclusive knots, matching dp_align_srvf's public
    return convention. q has endpoint-exclusive periodic samples.
    """
    q = np.asarray(q)
    gamma = np.clip(np.asarray(gamma), 0.0, 1.0)
    n, d = q.shape
    grid = np.linspace(0.0, 1.0, n, endpoint=False)
    grid_ext = np.r_[grid, 1.0]
    gamma_grid = np.linspace(0.0, 1.0, len(gamma))
    gamma_at_data = np.interp(grid, gamma_grid, gamma)

    q_warp = np.zeros_like(q)
    for j in range(d):
        q_warp[:, j] = np.interp(
            gamma_at_data, grid_ext, np.r_[q[:, j], q[0, j]]
        )

    gamma_dot_knots = np.gradient(gamma, gamma_grid)
    gamma_dot = np.interp(grid, gamma_grid, gamma_dot_knots)
    gamma_dot = np.maximum(gamma_dot, 1e-8)
    q_warp *= np.sqrt(gamma_dot)[:, None]
    return normalize_srvf(q_warp)


def optimal_rotation_2d_from_srvf(q, q_ref):
    M = q[:, :2].T @ q_ref[:, :2]
    U, _, Vt = np.linalg.svd(M)

    R = U @ Vt

    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R

def dp_align_srvf(q_ref, q, lam=0.0):
    """Discrete quadrature approximation to L2 alignment plus lambda*P(gamma)."""
    return _srvf_numerics.penalized_dp(
        q_ref, q, lam, endpoint_duplicated=False
    )

def center_beta(beta):
    return beta - beta.mean(axis=0, keepdims=True)


def center_embedding(X):
    Xc = X.copy()
    Xc[:, :2] = Xc[:, :2] - Xc[:, :2].mean(axis=0, keepdims=True)
    return Xc




def run_alignment(beta_raw, r_raw, X_raw, lambda_warp):
    K = len(X_raw)

    beta_warped = beta_raw.copy()
    r_warped = r_raw.copy()
    X_warped = X_raw.copy()

    beta_aligned = beta_raw.copy()
    r_aligned = r_raw.copy()
    X_aligned = X_raw.copy()
    q_aligned_store = np.zeros_like(X_raw)

    gammas = np.tile(np.linspace(0.0, 1.0, N), (K, 1))

    X_template = center_embedding(X_raw[0].copy())
    q_template = normalize_srvf(srvf(X_template))

    for it in range(N_ITER_ALIGN):
        print(f"      alignment iteration {it + 1}/{N_ITER_ALIGN}", flush=True)

        # Retain the actual sphere mean; its inverse is for cyclic initialization.
        beta_template = X_template[:, :2]

        for i in range(K):
            if i % 25 == 0:
                print(f"        aligned {i}/{K}", flush=True)

            beta_i = center_beta(beta_raw[i].copy())
            r_i = r_raw[i].copy()
            X_i = build_embedding(beta_i, r_i)

            if USE_CYCLIC_PREALIGNMENT:
                sh = best_cyclic_shift(X_template, X_i)
                beta_i = np.roll(beta_i, sh, axis=0)
                r_i = np.roll(r_i, sh)
                X_i = build_embedding(beta_i, r_i)

            q_i = normalize_srvf(srvf(X_i))
            gamma_i = dp_align_srvf(q_template, q_i, lam=lambda_warp)

            beta_w = interp_periodic(beta_i, gamma_i)
            r_w = interp_periodic(r_i, gamma_i)

            beta_w = center_beta(beta_w)
            X_w = build_embedding(beta_w, r_w)

            q_w = apply_gamma_to_srvf(q_i, gamma_i)

            R_i = optimal_rotation_2d_from_srvf(q_w, q_template)

            q_a = q_w.copy()
            q_a[:, :2] = q_w[:, :2] @ R_i
            q_a = normalize_srvf(q_a)

            beta_a = beta_w @ R_i
            beta_a = center_beta(beta_a)

            X_a = build_embedding(beta_a, r_w)


            beta_warped[i] = beta_w
            r_warped[i] = r_w
            X_warped[i] = X_w

            beta_aligned[i] = beta_a
            r_aligned[i] = r_w
            X_aligned[i] = X_a
            q_aligned_store[i] = q_a
            gammas[i] = gamma_i

        q_template = karcher_mean_sphere(
            q_aligned_store,
            max_iter=100,
            tol=1e-6,
            verbose=True
        )

        X_template = inverse_srvf(
            q_template,
            x0=np.zeros(3)
        )

        X_template = center_embedding(X_template)

    return (
        beta_warped,
        r_warped,
        X_warped,
        beta_aligned,
        r_aligned,
        X_aligned,
        gammas,
        q_aligned_store,
        q_template,
        X_template
    )

def align_to_fixed_template(beta_raw, r_raw, X_raw, X_template, lambda_warp, q_reference=None, return_srvfs=False):
    K = len(X_raw)

    beta_warped = beta_raw.copy()
    r_warped = r_raw.copy()
    X_warped = X_raw.copy()

    beta_aligned = beta_raw.copy()
    r_aligned = r_raw.copy()
    X_aligned = X_raw.copy()

    gammas = np.tile(np.linspace(0.0, 1.0, N), (K, 1))

    X_template_c = center_embedding(X_template.copy())
    beta_template = X_template_c[:, :2]
    q_template = normalize_srvf(srvf(X_template_c))
    if q_reference is not None:
        q_template = normalize_srvf(np.asarray(q_reference).copy())
    q_store = np.zeros_like(X_raw)

    for i in range(K):
        if i % 25 == 0:
            print(f"        fixed-template aligned {i}/{K}", flush=True)

        beta_i = center_beta(beta_raw[i].copy())
        r_i = r_raw[i].copy()
        X_i = build_embedding(beta_i, r_i)

        if USE_CYCLIC_PREALIGNMENT:
            sh = best_cyclic_shift(X_template_c, X_i)
            beta_i = np.roll(beta_i, sh, axis=0)
            r_i = np.roll(r_i, sh)
            X_i = build_embedding(beta_i, r_i)

        q_i = normalize_srvf(srvf(X_i))
        gamma_i = dp_align_srvf(q_template, q_i, lam=lambda_warp)

        beta_w = interp_periodic(beta_i, gamma_i)
        r_w = interp_periodic(r_i, gamma_i)

        beta_w = center_beta(beta_w)
        X_w = build_embedding(beta_w, r_w)

        q_w = apply_gamma_to_srvf(q_i, gamma_i)

        R_i = optimal_rotation_2d_from_srvf(q_w, q_template)

        q_a = q_w.copy()
        q_a[:, :2] = q_w[:, :2] @ R_i
        q_a = normalize_srvf(q_a)

        beta_a = beta_w @ R_i
        beta_a = center_beta(beta_a)

        X_a = build_embedding(beta_a, r_w)

        beta_warped[i] = beta_w
        r_warped[i] = r_w
        X_warped[i] = X_w

        beta_aligned[i] = beta_a
        r_aligned[i] = r_w
        X_aligned[i] = X_a
        gammas[i] = gamma_i
        q_store[i] = q_a

    result = (
        beta_warped,
        r_warped,
        X_warped,
        beta_aligned,
        r_aligned,
        X_aligned,
        gammas
    )
    return result + (q_store,) if return_srvfs else result

def gamma_to_psi(gamma, grid):
    gamma_dot = np.gradient(gamma, grid)
    gamma_dot = np.maximum(gamma_dot, 1e-8)
    psi = np.sqrt(gamma_dot)

    norm = np.sqrt(np.trapezoid(psi ** 2, grid))
    return psi / (norm + 1e-12)


def psi_to_gamma(psi, grid):
    density = psi ** 2
    gamma = np.zeros_like(grid)
    gamma[1:] = np.cumsum(
        0.5 * (density[:-1] + density[1:]) * np.diff(grid)
    )
    gamma = gamma / (gamma[-1] + 1e-12)
    gamma[0] = 0.0
    gamma[-1] = 1.0
    return gamma


def karcher_mean_gamma(gammas, grid):
    psi_all = np.array([
        gamma_to_psi(gammas[i], grid)
        for i in range(len(gammas))
    ])

    psi_all = psi_all[:, :, None]

    psi_mean = karcher_mean_sphere(
        psi_all,
        max_iter=100,
        tol=1e-6,
        verbose=False
    )

    psi_mean = psi_mean[:, 0]
    return psi_to_gamma(psi_mean, grid)
    
# ============================================================
# 9. Lambda tuning with training-set cross-validation logic
# ============================================================

def l2_curve_distance(A, B):
    return np.sqrt(np.mean(np.sum((A - B) ** 2, axis=1)))


def l2_function_distance(f, g):
    return np.sqrt(np.mean((f - g) ** 2))


def validation_shape_mse(beta_val_aligned, beta_template):
    return np.mean([
        np.mean(np.sum((beta_val_aligned[i] - beta_template) ** 2, axis=1))
        for i in range(len(beta_val_aligned))
    ])

def validation_thickness_mse(r_val_aligned, r_template):
    return np.mean([
        np.mean((r_val_aligned[i] - r_template) ** 2)
        for i in range(len(r_val_aligned))
    ])


def gamma_roughness_score(gammas):
    identity = np.linspace(0.0, 1.0, gammas.shape[1])
    deviation = gammas - identity[None, :]
    return np.mean(deviation ** 2)


def safe_zscore(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)

    if sd < 1e-12:
        return np.zeros_like(x)

    return (x - np.nanmean(x)) / sd


def srvf_sphere_geodesic_distance(q1, q2):
    """
    Geodesic distance between unit-normalized SRVFs on the Hilbert sphere.
    """
    ip = l2_inner(q1, q2)
    ip = np.clip(ip, -1.0, 1.0)
    return np.arccos(ip)

def srvf_norm(q):
    return np.sqrt(max(l2_inner(q, q), 0.0))


def log_map_sphere(q, mu):
    """
    Log map on the unit Hilbert sphere at mu, pointing to q.
    """
    ip = np.clip(l2_inner(mu, q), -1.0, 1.0)
    theta = np.arccos(ip)

    if theta < 1e-10:
        return np.zeros_like(mu)

    return (theta / np.sin(theta)) * (q - np.cos(theta) * mu)


def exp_map_sphere(v, mu):
    """
    Exp map on the unit Hilbert sphere at mu.
    """
    nv = srvf_norm(v)

    if nv < 1e-10:
        return mu.copy()

    return np.cos(nv) * mu + np.sin(nv) * (v / nv)
    
def karcher_mean_sphere(q_list, max_iter=100, tol=1e-6, verbose=False):
    mu = normalize_srvf(q_list[0].copy())
    Kloc = len(q_list)

    for it in range(max_iter):
        tangent_mean = np.zeros_like(mu)

        for q in q_list:
            qn = normalize_srvf(q)
            tangent_mean += log_map_sphere(qn, mu)

        tangent_mean /= Kloc

        mu_new = exp_map_sphere(tangent_mean, mu)
        mu_new = normalize_srvf(mu_new)

        diff = srvf_norm(mu_new - mu)

        if verbose:
            print(f"Karcher iteration {it + 1}: diff = {diff:.3e}")

        if diff < tol:
            mu = mu_new
            break

        mu = mu_new

    return mu

    
def validation_srvf_geodesic_mse_to_karcher(
    X_train_aligned,
    X_val_aligned,
    verbose=False
):
    q_train = np.array([
        normalize_srvf(srvf(X_train_aligned[i]))
        for i in range(len(X_train_aligned))
    ])

    q_val = np.array([
        normalize_srvf(srvf(X_val_aligned[i]))
        for i in range(len(X_val_aligned))
    ])

    q_mean = karcher_mean_sphere(
        q_train,
        max_iter=100,
        tol=1e-6,
        verbose=verbose
    )

    dists = np.array([
        srvf_sphere_geodesic_distance(q_val[i], q_mean)
        for i in range(len(q_val))
    ])

    return np.mean(dists ** 2), np.std(dists)
    

def gamma_deviation_mse(gammas):
    identity = np.linspace(0.0, 1.0, gammas.shape[1])
    return np.mean((gammas - identity[None, :]) ** 2)

def validation_srvf_geodesic_mse_to_fixed_mean(q_mean, X_val_aligned, q_val_aligned=None):
    q_val = np.array([
        normalize_srvf(srvf(X_val_aligned[i]))
        for i in range(len(X_val_aligned))
    ])

    if q_val_aligned is not None:
        q_val = q_val_aligned

    dists = np.array([
        srvf_sphere_geodesic_distance(q_val[i], q_mean)
        for i in range(len(q_val))
    ])

    return np.mean(dists ** 2), np.std(dists)
    
def select_lambda_5fold_cv_real(beta_raw, r_raw, X_raw, repeat_id=0):
    """
    Select gamma-warping lambda by K-fold CV on the training set.

    For each lambda and fold:
        1. Align training-fold curves.
        2. Build the training-fold aligned template.
        3. Align validation-fold curves to this template.
        4. Compute the Karcher mean of training-fold aligned SRVFs.
        5. Evaluate validation SRVF geodesic MSE to that mean.
    """

    rows = []

    kf = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE + repeat_id
    )

    for lam in LAMBDA_GRID:
        print(f"    lambda={lam}")

        for fold_id, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(X_raw)))):
            print(f"        fold {fold_id + 1}/{N_FOLDS}")

            beta_train = beta_raw[train_idx]
            r_train = r_raw[train_idx]
            X_train = X_raw[train_idx]

            beta_val = beta_raw[val_idx]
            r_val = r_raw[val_idx]
            X_val = X_raw[val_idx]


            (
                beta_train_w,
                r_train_w,
                X_train_w,
                beta_train_a,
                r_train_a,
                X_train_a,
                gam_train,
                q_train_a,
                q_mean,
                X_template
            ) = run_alignment(
                beta_train,
                r_train,
                X_train,
                lambda_warp=lam
            )


            (
                beta_val_w,
                r_val_w,
                X_val_w,
                beta_val_a,
                r_val_a,
                X_val_a,
                gam_val, q_val_a
            ) = align_to_fixed_template(
                beta_val,
                r_val,
                X_val,
                X_template=X_template,
                lambda_warp=lam, q_reference=q_mean, return_srvfs=True
            )

            verbose_flag = (repeat_id == 0 and fold_id == 0)

            srvf_geo_mse, srvf_geo_sd = validation_srvf_geodesic_mse_to_fixed_mean(
                q_mean=q_mean,
                X_val_aligned=X_val_a, q_val_aligned=q_val_a
            )


            rows.append({
                "repeat": repeat_id,
                "lambda": lam,
                "fold": fold_id,
                "srvf_geodesic_validation_mse": srvf_geo_mse,
                "srvf_geodesic_validation_sd": srvf_geo_sd,
                "mean_gamma_deviation_mse": gamma_deviation_mse(gam_val)
            })

    cv_table = pd.DataFrame(rows)

    cv_summary = (
        cv_table
        .groupby("lambda")["srvf_geodesic_validation_mse"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={
            "mean": "cv_srvf_geodesic_mse_mean",
            "std": "cv_srvf_geodesic_mse_sd"
        })
    )

    best_idx = cv_summary["cv_srvf_geodesic_mse_mean"].idxmin()
    best_lambda = cv_summary.loc[best_idx, "lambda"]

    print(
        "    selected lambda =",
        best_lambda,
        "| CV SRVF geodesic MSE =",
        cv_summary.loc[best_idx, "cv_srvf_geodesic_mse_mean"]
    )

    cv_table["best_lambda"] = best_lambda
    cv_summary["best_lambda"] = best_lambda

    return best_lambda, cv_table, cv_summary

def karcher_template_from_aligned_X(X_aligned, x0=None):
    """
    Compute SRVF Hilbert-sphere Karcher template from aligned embedded curves.

    Returns:
        X_template : inverse-SRVF reconstruction of the Karcher mean
        q_mean     : Karcher mean SRVF
        q_aligned  : aligned normalized SRVFs
    """
    q_aligned = np.array([
        normalize_srvf(srvf(X_aligned[i]))
        for i in range(len(X_aligned))
    ])

    q_mean = karcher_mean_sphere(
        q_aligned,
        max_iter=100,
        tol=1e-6,
        verbose=True
    )

    if x0 is None:
        x0 = np.zeros(q_mean.shape[1])

    X_template = inverse_srvf(q_mean, x0=x0)

    # center geometry only
    X_template = center_embedding(X_template)

    return X_template, q_mean, q_aligned


    
def compute_final_penalized_karcher_mean(X_aligned, gammas, lambda_star):
    q_aligned = np.array([
        normalize_srvf(srvf(X_aligned[i]))
        for i in range(len(X_aligned))
    ])

    q_mean_lambda = karcher_mean_sphere(
        q_aligned,
        max_iter=100,
        tol=1e-6,
        verbose=True
    )

    distances = np.array([
        srvf_sphere_geodesic_distance(q_aligned[i], q_mean_lambda)
        for i in range(len(q_aligned))
    ])

    penalties = np.array([
        gamma_deviation_mse(gammas[i][None, :])
        for i in range(len(gammas))
    ])

    summary = {
        "lambda_star": lambda_star,
        "mean_srvf_geodesic_distance": distances.mean(),
        "sd_srvf_geodesic_distance": distances.std(),
        "mean_warping_penalty": penalties.mean(),
        "sd_warping_penalty": penalties.std()
    }

    return q_mean_lambda, q_aligned, distances, penalties, summary
    

def compute_final_penalized_karcher_mean_from_q(q_aligned, gammas, lambda_star):
    q_aligned = np.array([
        normalize_srvf(q_aligned[i])
        for i in range(len(q_aligned))
    ])

    q_mean_lambda = karcher_mean_sphere(
        q_aligned,
        max_iter=100,
        tol=1e-6,
        verbose=True
    )

    distances = np.array([
        srvf_sphere_geodesic_distance(q_aligned[i], q_mean_lambda)
        for i in range(len(q_aligned))
    ])

    penalties = np.array([
        gamma_deviation_mse(gammas[i][None, :])
        for i in range(len(gammas))
    ])

    summary = {
        "lambda_star": lambda_star,
        "mean_srvf_geodesic_distance": distances.mean(),
        "sd_srvf_geodesic_distance": distances.std(),
        "mean_warping_penalty": penalties.mean(),
        "sd_warping_penalty": penalties.std()
    }

    return q_mean_lambda, q_aligned, distances, penalties, summary

    
def tune_regression_model_cv(spec, X_train, y_train, n_folds=N_FOLDS, fold_features=None):
    """Tune on fold-fitted representations, then refit on external training data."""
    if fold_features is not None and len(fold_features) != min(n_folds, len(y_train)):
        raise ValueError('Prepared feature folds do not match n_folds.')
    return _camus_cv.tune_supervised(spec, X_train, y_train, fold_features, 'regression')


def tune_classification_model_cv(spec, X_train, y_train, n_folds=N_FOLDS, fold_features=None):
    """Tune on fold-fitted representations, then refit on external training data."""
    if fold_features is not None and len(fold_features) != min(n_folds, len(y_train)):
        raise ValueError('Prepared feature folds do not match n_folds.')
    return _camus_cv.tune_supervised(spec, X_train, y_train, fold_features, 'classification')

def select_k_train_cv(X_train, n_folds=N_FOLDS, fold_features=None):
    """Select clustering using fold-local features and fold-local scaling."""
    if fold_features is not None and len(fold_features) != min(n_folds, len(X_train)):
        raise ValueError('Prepared feature folds do not match n_folds.')
    return _camus_cv.select_clusters(X_train, fold_features, clustering_model_specs())
    
# ============================================================
# 10. Feature extraction
# ============================================================

def flatten_curves(A):
    return A.reshape(A.shape[0], -1)


def extract_scalar_features(beta_all, r_all, X_all, X_template):
    q_template = normalize_srvf(srvf(X_template))

    rows = []

    for i in range(len(beta_all)):
        beta = beta_all[i]
        r = r_all[i]
        X = X_all[i]
        q = normalize_srvf(srvf(X))

        rows.append({
            "area": curve_area(beta),
            "perimeter": curve_perimeter(beta),
            "compactness": compactness(beta),
            "mean_thickness": np.mean(r),
            "sd_thickness": np.std(r),
            "min_thickness": np.min(r),
            "max_thickness": np.max(r),
            "range_thickness": np.max(r) - np.min(r),
            "l2_thickness": np.sqrt(np.sum(r ** 2) * dt),
            "shape_distance_to_template": l2_curve_distance(beta, X_template[:, :2]),
            "srvf_distance_to_template": np.sqrt(np.sum((q - q_template) ** 2) * dt)
        })

    return pd.DataFrame(rows)


def fit_pca_features_train_test(train_arr, test_arr, prefix):
    Xtr = flatten_curves(train_arr)
    Xte = flatten_curves(test_arr)

    n_comp = min(N_PCA_COMPONENTS, Xtr.shape[0] - 1, Xtr.shape[1])
    n_comp = max(1, n_comp)

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    Ztr = pca.fit_transform(Xtr_s)
    Zte = pca.transform(Xte_s)

    cols = [f"{prefix}_PC{k + 1}" for k in range(n_comp)]

    return (
        pd.DataFrame(Ztr, columns=cols),
        pd.DataFrame(Zte, columns=cols)
    )


def build_features_train_test(
    beta_train, r_train, X_train,
    beta_test, r_test, X_test,
    X_template
):
    scalar_train = extract_scalar_features(beta_train, r_train, X_train, X_template)
    scalar_test = extract_scalar_features(beta_test, r_test, X_test, X_template)

    shape_train, shape_test = fit_pca_features_train_test(
        beta_train, beta_test, "shape"
    )

    thick_train, thick_test = fit_pca_features_train_test(
        r_train[:, :, None], r_test[:, :, None], "thickness"
    )

    srvf_train = np.array([
        normalize_srvf(srvf(X_train[i]))
        for i in range(len(X_train))
    ])

    srvf_test = np.array([
        normalize_srvf(srvf(X_test[i]))
        for i in range(len(X_test))
    ])

    srvf_train_df, srvf_test_df = fit_pca_features_train_test(
        srvf_train, srvf_test, "srvf"
    )

    embed_train, embed_test = fit_pca_features_train_test(
        X_train, X_test, "embed"
    )

    Xtrain = pd.concat(
        [scalar_train, shape_train, thick_train, srvf_train_df, embed_train],
        axis=1
    )

    Xtest = pd.concat(
        [scalar_test, shape_test, thick_test, srvf_test_df, embed_test],
        axis=1
    )

    Xtrain = Xtrain.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    Xtest = Xtest.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return Xtrain, Xtest


# ============================================================
# 11. ML model definitions
# ============================================================

def regression_model_specs():
    specs = {
        "linear": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LinearRegression())
            ]),
            "params": {}
        },
        "ridge": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge())
            ]),
            "params": {
                "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
            }
        },
        "lasso": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Lasso(max_iter=10000))
            ]),
            "params": {
                "model__alpha": [0.001, 0.01, 0.1, 1.0]
            }
        },
        "elasticnet": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", ElasticNet(max_iter=10000))
            ]),
            "params": {
                "model__alpha": [0.001, 0.01, 0.1, 1.0],
                "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
            }
        },
        "random_forest": {
            "model": RandomForestRegressor(random_state=RANDOM_STATE),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [None, 3, 5, 10],
                "min_samples_leaf": [1, 3, 5]
            }
        },
        "svr": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVR())
            ]),
            "params": {
                "model__C": [0.1, 1.0, 10.0],
                "model__epsilon": [0.01, 0.1, 1.0],
                "model__kernel": ["rbf", "linear"]
            }
        }
    }

    if HAS_XGBOOST:
        specs["xgboost"] = {
            "model": XGBRegressor(
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                verbosity=0
            ),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [2, 3, 5],
                "learning_rate": [0.01, 0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0]
            }
        }

    if HAS_LIGHTGBM:
        specs["lightgbm"] = {
            "model": LGBMRegressor(
                random_state=RANDOM_STATE,
                verbose=-1
            ),
            "params": {
                "n_estimators": [100, 300],
                "num_leaves": [7, 15, 31],
                "learning_rate": [0.01, 0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0]
            }
        }

    return specs


def classification_model_specs():
    specs = {
        "logistic": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=5000))
            ]),
            "params": {
                "model__C": [0.01, 0.1, 1.0, 10.0]
            }
        },
        "elasticnet_logistic": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    max_iter=5000
                ))
            ]),
            "params": {
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
            }
        },
        "random_forest": {
            "model": RandomForestClassifier(random_state=RANDOM_STATE),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [None, 3, 5, 10],
                "min_samples_leaf": [1, 3, 5]
            }
        },
        "svm": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(probability=True))
            ]),
            "params": {
                "model__C": [0.1, 1.0, 10.0],
                "model__kernel": ["rbf", "linear"],
                "model__gamma": ["scale", "auto"]
            }
        }
    }

    if HAS_XGBOOST:
        specs["xgboost"] = {
            "model": XGBClassifier(
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                verbosity=0
            ),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [2, 3, 5],
                "learning_rate": [0.01, 0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0]
            }
        }

    if HAS_LIGHTGBM:
        specs["lightgbm"] = {
            "model": LGBMClassifier(
                random_state=RANDOM_STATE,
                verbose=-1
            ),
            "params": {
                "n_estimators": [100, 300],
                "num_leaves": [7, 15, 31],
                "learning_rate": [0.01, 0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0]
            }
        }

    return specs


# ============================================================
# 12. Manual validation tuning
# ============================================================

def tune_regression_model(spec, X_train, y_train, X_val, y_val):
    best_model = None
    best_params = {}
    best_score = np.inf

    grid = list(ParameterGrid(spec["params"])) if spec["params"] else [{}]

    for params in grid:
        model = spec["model"]
        model.set_params(**params)
        model.fit(X_train, y_train)

        pred_val = model.predict(X_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))

        if rmse_val < best_score:
            best_score = rmse_val
            best_params = params
            best_model = model

    return best_model, best_params, best_score


def tune_classification_model(spec, X_train, y_train, X_val, y_val):
    best_model = None
    best_params = {}
    best_score = -np.inf

    grid = list(ParameterGrid(spec["params"])) if spec["params"] else [{}]

    for params in grid:
        model = spec["model"]
        model.set_params(**params)
        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            prob_val = model.predict_proba(X_val)[:, 1]
            try:
                score = roc_auc_score(y_val, prob_val)
            except Exception:
                score = balanced_accuracy_score(y_val, model.predict(X_val))
        else:
            score = balanced_accuracy_score(y_val, model.predict(X_val))

        if score > best_score:
            best_score = score
            best_params = params
            best_model = model

    return best_model, best_params, best_score


def evaluate_regression(pred, y):
    pred = np.asarray(pred, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(y) >= 2 and np.std(y) > 1e-12 and np.std(pred) > 1e-12:
        corr = np.corrcoef(y, pred)[0, 1]
    else:
        corr = np.nan

    return {
        "mae": mean_absolute_error(y, pred),
        "rmse": np.sqrt(mean_squared_error(y, pred)),
        "r2": r2_score(y, pred) if len(y) >= 2 else np.nan,
        "corr": corr
    }


def evaluate_classification(model, X, y):
    pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X)[:, 1]
        try:
            auc = roc_auc_score(y, prob)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "f1": f1_score(y, pred, zero_division=0),
        "auc": auc,
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan
    }


# ============================================================
# 13. Alignment evaluation
# ============================================================

def evaluate_alignment_split(beta_raw, r_raw, X_raw, beta_a, r_a, X_a, X_template):
    beta_template = X_template[:, :2]
    r_template = X_template[:, 2] / ALPHA_EMBED
    q_template = normalize_srvf(srvf(X_template))

    shape_before = []
    shape_after = []
    thick_before = []
    thick_after = []
    srvf_before = []
    srvf_after = []

    for i in range(len(beta_raw)):
        shape_before.append(l2_curve_distance(beta_raw[i], beta_template))
        shape_after.append(l2_curve_distance(beta_a[i], beta_template))

        thick_before.append(l2_function_distance(r_raw[i], r_template))
        thick_after.append(l2_function_distance(r_a[i], r_template))

        q_raw_i = normalize_srvf(srvf(X_raw[i]))
        q_a_i = normalize_srvf(srvf(X_a[i]))

        srvf_before.append(np.sqrt(np.sum((q_raw_i - q_template) ** 2) * dt))
        srvf_after.append(np.sqrt(np.sum((q_a_i - q_template) ** 2) * dt))

    return {
        "shape_before": np.mean(shape_before),
        "shape_after": np.mean(shape_after),
        "shape_change_percent": 100 * (np.mean(shape_before) - np.mean(shape_after)) / np.mean(shape_before),
        "thickness_before": np.mean(thick_before),
        "thickness_after": np.mean(thick_after),
        "thickness_change_percent": 100 * (np.mean(thick_before) - np.mean(thick_after)) / np.mean(thick_before),
        "srvf_before": np.mean(srvf_before),
        "srvf_after": np.mean(srvf_after),
        "srvf_change_percent": 100 * (np.mean(srvf_before) - np.mean(srvf_after)) / np.mean(srvf_before)
    }


# ============================================================
# 14. Figures
# ============================================================

def plot_lambda_summary(lambda_results):
    counts = lambda_results["best_lambda"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_xlabel(r"Selected $\lambda$")
    ax.set_ylabel("Frequency")
    ax.set_title("Distribution of selected lambda over repetitions")

    plt.tight_layout()

    save_figure(
        fig,
        "fig_selected_lambda_distribution",
        caption=(
            "Distribution of selected warping regularization parameters over repeated "
            "training-set cross-validation splits."
        ),
        label="fig:selected_lambda_distribution"
    )

    plt.close(fig)


def plot_population_mean_contours(beta_aligned, r_aligned, X_template):
    """
    Plot the population mean using the SRVF Karcher template.
    No Euclidean mean is used for the displayed mean contours.
    """

    beta_mean = X_template[:, :2]
    r_mean = X_template[:, 2] / ALPHA_EMBED
    beta_out_mean = reconstruct_outer(beta_mean, r_mean)

    fig, ax = plt.subplots(figsize=(8, 8))

    for i in range(len(beta_aligned)):
        B = close_for_plot(beta_aligned[i])
        ax.plot(B[:, 0], B[:, 1], color="lightgray", alpha=0.45, lw=1)

    B_in = close_for_plot(beta_mean)
    B_out = close_for_plot(beta_out_mean)

    ax.plot(B_in[:, 0], B_in[:, 1], lw=4, label="Karcher mean endocardium")
    ax.plot(B_out[:, 0], B_out[:, 1], lw=4, linestyle="--", label="Karcher mean epicardium")

    ax.set_title("Population SRVF Karcher mean ventricular contours")
    ax.axis("equal")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        "fig_population_mean_ventricular_contours",
        caption=(
            "Population ventricular contours after embedded SRVF alignment. "
            "The displayed mean contours are reconstructed from the Hilbert-sphere "
            "SRVF Karcher mean."
        ),
        label="fig:population_mean_contours"
    )

    plt.close(fig)


def plot_thickness_modes(r_aligned):
    r_mean = np.mean(r_aligned, axis=0)
    r_centered = r_aligned - r_mean

    U, S, Vt = np.linalg.svd(r_centered, full_matrices=False)

    n_modes = min(3, Vt.shape[0])

    fig, axes = plt.subplots(n_modes, 1, figsize=(10, 4 * n_modes), sharex=True)

    if n_modes == 1:
        axes = [axes]

    for k in range(n_modes):
        pc = Vt[k]
        sd = np.std(U[:, k] * S[k])

        scale_thickness = 0.45

        r_minus = r_mean - scale_thickness * 2 * sd * pc
        r_plus = r_mean + scale_thickness * 2 * sd * pc

        r_minus = gaussian_filter1d(r_minus, sigma=1.5)
        r_plus = gaussian_filter1d(r_plus, sigma=1.5)

        ax = axes[k]

        for i in range(len(r_aligned)):
            ax.plot(s, r_aligned[i], color="lightgray", alpha=0.35, lw=1)

        ax.plot(s, r_mean, "k", lw=3, label="Karcher mean thickness")
        ax.plot(s, r_minus, "--", lw=2.5, label="-2 SD")
        ax.plot(s, r_plus, "--", lw=2.5, label="+2 SD")

        ax.set_title(f"Thickness variation along PC{k + 1}")
        ax.set_ylabel("Thickness")
        ax.legend()

    axes[-1].set_xlabel("Normalized contour parameter")

    plt.tight_layout()

    save_figure(
        fig,
        "fig_thickness_modes_with_raw",
        caption=(
            "Dominant modes of myocardial thickness variation after alignment."
        ),
        label="fig:thickness_modes"
    )

    plt.close(fig)



def plot_clean_real_data_pipeline(
    beta_in_raw,
    beta_out_raw,
    q_raw,
    q_hat,
    beta_warped,
    beta_aligned,
    r_aligned,
    gammas,
    X_template
):
    """
    Clean publication-ready pipeline figure.

    Improvements:
    1. Step 1--3 simplified.
    2. SRVF magnitude panels show mean +/- 2 SD only.
    3. Warping panel shows deviation from identity.
    4. Aligned contours emphasize mean shape.
    5. Mean reconstruction includes myocardial shading.
    """

    K = len(beta_in_raw)

    

    # SRVF Karcher template
    beta_mean_template = enforce_curve_closure(X_template[:, :2])
    r_mean = enforce_scalar_periodicity(X_template[:, 2] / ALPHA_EMBED)

    # Step 2 mean: raw CAMUS coordinate system
    beta_mean_raw = beta_mean_template.copy()
    beta_mean_raw = beta_mean_raw + (
        np.mean(beta_in_raw.reshape(-1, 2), axis=0, keepdims=True)
        - np.mean(beta_mean_raw, axis=0, keepdims=True)
    )

    # Step 9 mean: aligned coordinate system
    beta_mean_aligned = beta_mean_template.copy()
    beta_mean_aligned = beta_mean_aligned + (
        np.mean(beta_aligned.reshape(-1, 2), axis=0, keepdims=True)
        - np.mean(beta_mean_aligned, axis=0, keepdims=True)
    )

    beta_out_mean_raw = reconstruct_outer(beta_mean_raw, r_mean)
    beta_out_mean_aligned = reconstruct_outer(beta_mean_aligned, r_mean)

    # --------------------------------------------------------
    # SRVF summaries
    # --------------------------------------------------------
    q_raw_mag = np.linalg.norm(q_raw, axis=2)
    q_hat_mag = np.linalg.norm(q_hat, axis=2)

    q_raw_mean = q_raw_mag.mean(axis=0)
    q_raw_sd = q_raw_mag.std(axis=0)

    q_hat_mean = q_hat_mag.mean(axis=0)
    q_hat_sd = q_hat_mag.std(axis=0)

    # --------------------------------------------------------
    # Warping deviation from identity
    # --------------------------------------------------------
    identity = np.linspace(0.0, 1.0, gammas.shape[1])
    gamma_dev = gammas - identity[None, :]

    gamma_karcher_mean = karcher_mean_gamma(gammas, identity)
    gamma_karcher_dev = gamma_karcher_mean - identity
    gamma_sd = gamma_dev.std(axis=0)

    # --------------------------------------------------------
    # Figure layout
    # --------------------------------------------------------
    fig, axes = plt.subplots(3, 3, figsize=(16, 13))

    from matplotlib.lines import Line2D

    # --------------------------------------------------------
    # Colors
    # --------------------------------------------------------
    col_inner = "#1f5fd6"
    col_outer = "#d62728"
    col_mean = "#111111"
    col_gray = "0.55"
    col_warp = "#1b7f2a"
    col_srvf = "#174a7e"
    col_norm = "#d95f02"
    col_thick = "#1f5fd6"
    col_after_warp = "#7b1fa2"
    col_aligned = "#8b0000"

    def clean_axis(ax, equal=False):
        ax.grid(True, linestyle="--", alpha=0.20)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if equal:
            ax.axis("equal")
            ax.set_xlabel(r"$\beta_x(s)$")
            ax.set_ylabel(r"$\beta_y(s)$")

    # ========================================================
    # Step 1: raw ventricular contours
    # ========================================================
    ax = axes[0, 0]

    for i in range(K):

        B_in = close_for_plot(beta_in_raw[i])
        B_out = close_for_plot(beta_out_raw[i])

        ax.plot(
            B_in[:, 0],
            B_in[:, 1],
            color=col_inner,
            alpha=0.30,
            lw=1.0
        )

        ax.plot(
            B_out[:, 0],
            B_out[:, 1],
            color=col_outer,
            alpha=0.30,
            lw=1.0
        )

    ax.set_title("Raw myocardial contours")
    clean_axis(ax, equal=True)

    inner_handle = Line2D(
        [0], [0],
        color=col_inner,
        lw=2.5,
        alpha=1.0,
        label="Endocardial contours"
    )

    outer_handle = Line2D(
        [0], [0],
        color=col_outer,
        lw=2.5,
        alpha=1.0,
        label="Epicardial contours"
    )

    ax.legend(
        handles=[inner_handle, outer_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 2: embedded curves, XY projection
    # ========================================================
    ax = axes[0, 1]

    for i in range(K):

        B = close_for_plot(beta_in_raw[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_gray,
            alpha=0.28,
            lw=0.9
        )

    B_template = close_for_plot(beta_mean_raw)

    ax.plot(
        B_template[:, 0],
        B_template[:, 1],
        color=col_mean,
        lw=3.0
    )

    ax.set_title("Embedded contour projections")
    clean_axis(ax, equal=True)

    raw_proj_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        label=r"Raw embedded projections"
    )

    template_handle = Line2D(
        [0], [0],
        color=col_mean,
        lw=3.0,
        label=r"SRVF Karcher mean projection $\bar{\beta}(s)$"
    )

    ax.legend(
        handles=[raw_proj_handle, template_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 3: warping deviation from identity
    # ========================================================
    ax = axes[0, 2]

    for i in range(K):

        ax.plot(
            s,
            gamma_dev[i],
            color=col_gray,
            alpha=0.24,
            lw=0.9
        )

    ax.fill_between(
        s,
        gamma_karcher_dev - 2 * gamma_sd,
        gamma_karcher_dev + 2 * gamma_sd,
        color=col_warp,
        alpha=0.16
    )

    ax.plot(
        s,
        gamma_karcher_dev,
        color=col_warp,
        lw=3.0
    )

    ax.axhline(
        0.0,
        color="black",
        linestyle="--",
        lw=1.6
    )

    ax.set_title("Warping deviations from identity")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\gamma_i(s)-s$")

    clean_axis(ax)

    warp_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        label=r"Individual warping deviations"
    )

    warp_sd_handle = Line2D(
        [0], [0],
        color=col_warp,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    warp_mean_handle = Line2D(
        [0], [0],
        color=col_warp,
        lw=3.0,
        label=r"Karcher mean deviation"
    )

    ax.legend(
        handles=[warp_handle, warp_sd_handle, warp_mean_handle],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    max_dev = np.nanmax(np.abs(gamma_dev))

    if np.isfinite(max_dev) and max_dev > 0:

        ax.set_ylim(
            -1.15 * max_dev,
             1.15 * max_dev
        )

    # ========================================================
    # Step 4: raw SRVF magnitude
    # ========================================================
    ax = axes[1, 0]

    for i in range(K):

        ax.plot(
            s,
            q_raw_mag[i],
            color=col_gray,
            alpha=0.24,
            lw=0.8
        )

    ax.fill_between(
        s,
        q_raw_mean - 2 * q_raw_sd,
        q_raw_mean + 2 * q_raw_sd,
        color=col_srvf,
        alpha=0.12
    )

    ax.plot(
        s,
        q_raw_mean,
        color=col_srvf,
        lw=3.0
    )

    ax.set_title("Raw SRVF magnitudes")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\|q_i(s)\|$")

    clean_axis(ax)

    raw_srvf_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        label=r"Raw SRVF magnitudes"
    )

    raw_srvf_sd_handle = Line2D(
        [0], [0],
        color=col_srvf,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    raw_srvf_mean_handle = Line2D(
        [0], [0],
        color=col_srvf,
        lw=3.0,
        label=r"Pointwise mean magnitude"
    )

    ax.legend(
        handles=[
            raw_srvf_handle,
            raw_srvf_sd_handle,
            raw_srvf_mean_handle
        ],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 5: normalized SRVF magnitude
    # ========================================================
    ax = axes[1, 1]

    for i in range(K):

        ax.plot(
            s,
            q_hat_mag[i],
            color=col_gray,
            alpha=0.22,
            lw=0.8
        )

    ax.fill_between(
        s,
        q_hat_mean - 2 * q_hat_sd,
        q_hat_mean + 2 * q_hat_sd,
        color=col_norm,
        alpha=0.14
    )

    ax.plot(
        s,
        q_hat_mean,
        color=col_norm,
        lw=3.0
    )

    ax.set_title("Normalized SRVF magnitudes")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\|\tilde q_i(s)\|$")

    clean_axis(ax)

    norm_srvf_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        label=r"Normalized SRVF magnitudes"
    )

    norm_srvf_sd_handle = Line2D(
        [0], [0],
        color=col_norm,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    norm_srvf_mean_handle = Line2D(
        [0], [0],
        color=col_norm,
        lw=3.0,
        label=r"Pointwise mean magnitude"
    )

    ax.legend(
        handles=[
            norm_srvf_handle,
            norm_srvf_sd_handle,
            norm_srvf_mean_handle
        ],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 6: SRVF-aligned thickness component
    # ========================================================
   

    ax = axes[1, 2]

    X_aligned_display = np.array([
        build_embedding(beta_aligned[i], r_aligned[i])
        for i in range(K)
    ])

    r_display = np.array([
        enforce_scalar_periodicity(
            Xi[:, 2] / ALPHA_EMBED
        )
        for Xi in X_aligned_display
    ])

    r_sd = r_display.std(axis=0)
    
    for i in range(K):

        ax.plot(
            s,
            r_display[i],
            color=col_gray,
            alpha=0.50,
            lw=0.9
        )

    
    r_karcher = enforce_scalar_periodicity(
        X_template[:, 2] / ALPHA_EMBED
    )
    

    ax.fill_between(
        s,
        r_karcher - 2 * r_sd,
        r_karcher + 2 * r_sd,
        color=col_thick,
        alpha=0.14
    )

    ax.plot(
        s,
        r_karcher,
        color=col_thick,
        lw=3.0
    )

    ax.set_title(r"SRVF-aligned thickness component $r_{\mathrm{SRVF}}(s)$")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$r_{\mathrm{SRVF}}(s)$")

    clean_axis(ax)

    thick_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        label=r"Aligned thickness profiles $r_{\mathrm{SRVF},i}(s)$"
    )

    thick_sd_handle = Line2D(
        [0], [0],
        color=col_thick,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    thick_mean_handle = Line2D(
        [0], [0],
        color=col_thick,
        lw=3.0,
        label=r"Karcher mean thickness $\bar r_{\mathrm{SRVF}}(s)$"
    )

    ax.legend(
        handles=[
            thick_handle,
            thick_sd_handle,
            thick_mean_handle
        ],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 7: contours after embedded warping
    # ========================================================
    ax = axes[2, 0]

    for i in range(K):

        B = close_for_plot(beta_warped[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_after_warp,
            alpha=0.28,
            lw=0.85
        )

    ax.set_title("Contours after elastic warping")
    clean_axis(ax, equal=True)

    warped_handle = Line2D(
        [0], [0],
        color=col_after_warp,
        lw=2.5,
        label=r"Warped contours $\beta_i(\gamma_i(s))$"
    )

    ax.legend(
        handles=[warped_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 8: contours after SO(2) alignment
    # ========================================================
    ax = axes[2, 1]

    for i in range(K):

        B = close_for_plot(beta_aligned[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_aligned,
            alpha=0.28,
            lw=0.85
        )

    ax.set_title(r"Contours after $SO(2)$ alignment")
    clean_axis(ax, equal=True)

    aligned_handle = Line2D(
        [0], [0],
        color=col_aligned,
        lw=2.5,
        label=r"Aligned contours $R_i(\beta_i(\gamma_i(s)))$"
    )

    ax.legend(
        handles=[aligned_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 9: SRVF Karcher mean reconstruction
    # ========================================================
    ax = axes[2, 2]

    for i in range(K):

        B = close_for_plot(beta_aligned[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_gray,
            alpha=0.22,
            lw=0.75
        )

    B_in = close_for_plot(beta_mean_aligned)
    B_out = close_for_plot(beta_out_mean_aligned)

    poly_x = np.concatenate([
        B_out[:, 0],
        B_in[::-1, 0]
    ])

    poly_y = np.concatenate([
        B_out[:, 1],
        B_in[::-1, 1]
    ])

    ax.fill(
        poly_x,
        poly_y,
        color="0.80",
        alpha=0.55
    )

    ax.plot(
        B_in[:, 0],
        B_in[:, 1],
        color=col_inner,
        lw=3.2
    )

    ax.plot(
        B_out[:, 0],
        B_out[:, 1],
        color=col_outer,
        linestyle="--",
        lw=3.0
    )

    ax.set_title("SRVF Karcher mean reconstruction")
    clean_axis(ax, equal=True)

    raw_aligned_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.0,
        label=r"Aligned contours"
    )

    band_handle = Line2D(
        [0], [0],
        color="0.60",
        lw=6,
        alpha=0.55,
        label=r"Mean myocardium"
    )

    mean_inner_handle = Line2D(
        [0], [0],
        color=col_inner,
        lw=3.2,
        label=r"Mean inner contour $\bar{\beta}(s)$"
    )

    mean_outer_handle = Line2D(
        [0], [0],
        color=col_outer,
        linestyle="--",
        lw=3.0,
        label=r"Mean outer contour $\bar{\beta}_{\mathrm{out}}(s)$"
    )

    ax.legend(
        handles=[
            raw_aligned_handle,
            band_handle,
            mean_inner_handle,
            mean_outer_handle
        ],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    plt.tight_layout()

    save_figure(
        fig,
        "fig01_real_data_pipeline_clean",
        caption=(
            "Clean overview of the embedded SRVF real-data pipeline. "
            "Raw contours are transformed into an embedded curve representation, "
            "mapped to SRVF space, aligned by mild warping and rotational normalization, "
            "and reconstructed into mean endocardial and epicardial contours. "
            "Warping is shown as deviation from the identity map."
        ),
        label="fig:real_data_pipeline_clean"
    )

    plt.close(fig)

    
def plot_normal_ef_cluster_shapes(beta, r, labels, name):
    fig, ax = plt.subplots(figsize=(7, 7))

    for c in sorted(np.unique(labels)):
        idx = np.where(labels == c)[0]

        if len(idx) < 2:
            continue

        beta_mean, r_mean, X_mean = karcher_template_from_cluster(
            beta[idx],
            r[idx]
        )
        beta_out = reconstruct_outer(beta_mean, r_mean)

        for j in idx:
            B = close_for_plot(beta[j])
            ax.plot(B[:, 0], B[:, 1], color="lightgray", alpha=0.25, lw=1)

        B_in = close_for_plot(beta_mean)
        B_out = close_for_plot(beta_out)

        ax.plot(B_in[:, 0], B_in[:, 1], lw=4, label=f"Cluster {c} endocardium")
        ax.plot(B_out[:, 0], B_out[:, 1], "--", lw=3, label=f"Cluster {c} epicardium")

    ax.set_title("Normal EF structural clusters: mean shapes")
    ax.axis("equal")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Cluster-wise mean ventricular contours among patients with normal EF. "
            "Different clusters indicate structural heterogeneity despite preserved EF."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_normal_ef_cluster_thickness(beta, r, labels, name):
    fig, ax = plt.subplots(figsize=(9, 5))

    for c in sorted(np.unique(labels)):
        idx = np.where(labels == c)[0]

        if len(idx) < 2:
            continue

        beta_mean, r_mean, X_mean = karcher_template_from_cluster(
            beta[idx],
            r[idx]
        )

        r_sd = r[idx].std(axis=0)

        ax.plot(s, r_mean, lw=3, label=f"Cluster {c} Karcher mean")
        ax.fill_between(
            s,
            r_mean - r_sd,
            r_mean + r_sd,
            alpha=0.20
        )

    ax.set_title("Normal EF structural clusters: Karcher mean thickness profiles")
    ax.set_xlabel("Normalized contour parameter")
    ax.set_ylabel("Thickness")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Cluster-wise Karcher mean myocardial thickness profiles among patients "
            "with normal EF. Shaded regions represent pointwise plus/minus one "
            "standard deviation within each cluster."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)

    ax.set_title("Normal EF structural clusters: thickness profiles")
    ax.set_xlabel("Normalized contour parameter")
    ax.set_ylabel("Thickness")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Cluster-wise myocardial thickness profiles among patients with normal EF. "
            "Shaded regions represent plus/minus one standard deviation."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)

def plot_normal_ef_cluster_summary_figure(
    beta,
    r,
    labels,
    features=None,
    name="fig_normal_EF_cluster_summary"
):
    """
    Four-panel normal-EF structural cluster figure.

    A: Cluster-wise Karcher mean ventricular shapes
    B: Cluster-wise myocardial thickness profiles
    C: Cluster-wise embedded mean curves X(s)
    D: Cluster separation in PCA feature space
    """

    fig = plt.figure(figsize=(15, 10))

    ax_shape = fig.add_subplot(2, 2, 1)
    ax_thick = fig.add_subplot(2, 2, 2)
    ax_embed = fig.add_subplot(2, 2, 3, projection="3d")
    ax_pca = fig.add_subplot(2, 2, 4)

    colors = [
        "#1f77b4",
        "#d62728",
        "#2ca02c",
        "#9467bd",
        "#ff7f0e",
        "#8c564b"
    ]

    cluster_ids = sorted(np.unique(labels))

    for cc, c in enumerate(cluster_ids):

        idx = np.where(labels == c)[0]

        if len(idx) < 2:
            continue

        color = colors[cc % len(colors)]

        beta_mean, r_mean, X_mean = karcher_template_from_cluster(
            beta[idx],
            r[idx]
        )

        beta_out_mean = reconstruct_outer(beta_mean, r_mean)

        # ----------------------------------------------------
        # A. Shape
        # ----------------------------------------------------
        for j in idx:
            B = close_for_plot(beta[j])
            ax_shape.plot(
                B[:, 0],
                B[:, 1],
                color=color,
                alpha=0.08,
                lw=0.8
            )

        B_in = close_for_plot(beta_mean)
        B_out = close_for_plot(beta_out_mean)

        ax_shape.plot(
            B_in[:, 0],
            B_in[:, 1],
            color=color,
            lw=3.2,
            label=f"Cluster {c} endocardium"
        )

        ax_shape.plot(
            B_out[:, 0],
            B_out[:, 1],
            color=color,
            linestyle="--",
            lw=2.8,
            label=f"Cluster {c} epicardium"
        )

        # ----------------------------------------------------
        # B. Thickness
        # ----------------------------------------------------
        r_sd = np.std(r[idx], axis=0)

        for j in idx:
            ax_thick.plot(
                s,
                r[j],
                color=color,
                alpha=0.08,
                lw=0.8
            )

        ax_thick.fill_between(
            s,
            r_mean - r_sd,
            r_mean + r_sd,
            color=color,
            alpha=0.12
        )

        ax_thick.plot(
            s,
            r_mean,
            color=color,
            lw=3.0,
            label=f"Cluster {c}"
        )

        # ----------------------------------------------------
        # C. Embedded mean curve
        # ----------------------------------------------------
        Xc = build_embedding(beta_mean, r_mean)

        ax_embed.plot(
            Xc[:, 0],
            Xc[:, 1],
            Xc[:, 2],
            color=color,
            lw=3.0,
            label=f"Cluster {c}"
        )

        ax_embed.plot(
            Xc[:, 0],
            Xc[:, 1],
            zs=0,
            zdir="z",
            color=color,
            lw=1.8,
            alpha=0.35
        )

    # --------------------------------------------------------
    # D. PCA feature-space separation with centroids and ellipses
    # --------------------------------------------------------
    if features is not None:

        from matplotlib.patches import Ellipse

        def add_confidence_ellipse(ax, x, y, color, level=0.95):
            """
            Add an approximate 95% confidence ellipse for 2D points.
            For chi-square df=2, 95% quantile is 5.991.
            """
            x = np.asarray(x)
            y = np.asarray(y)

            if len(x) < 3:
                return

            cov = np.cov(x, y)

            if not np.all(np.isfinite(cov)):
                return

            vals, vecs = np.linalg.eigh(cov)
            order = vals.argsort()[::-1]

            vals = vals[order]
            vecs = vecs[:, order]

            vals = np.maximum(vals, 1e-10)

            chi2_95 = 5.991
            width, height = 2.0 * np.sqrt(vals * chi2_95)

            angle = np.degrees(
                np.arctan2(
                    vecs[1, 0],
                    vecs[0, 0]
                )
            )

            ell = Ellipse(
                xy=(np.mean(x), np.mean(y)),
                width=width,
                height=height,
                angle=angle,
                facecolor=color,
                edgecolor=color,
                alpha=0.15,
                lw=2.0
            )

            ax.add_patch(ell)

            ell_edge = Ellipse(
                xy=(np.mean(x), np.mean(y)),
                width=width,
                height=height,
                angle=angle,
                facecolor="none",
                edgecolor=color,
                alpha=0.85,
                lw=2.0
            )

            ax.add_patch(ell_edge)

        X = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        pca = PCA(
            n_components=2,
            random_state=RANDOM_STATE
        )

        Z = pca.fit_transform(Xs)

        explained = 100.0 * pca.explained_variance_ratio_

        for cc, c in enumerate(cluster_ids):

            idx = np.where(labels == c)[0]

            if len(idx) == 0:
                continue

            color = colors[cc % len(colors)]

            # Individual patients
            ax_pca.scatter(
                Z[idx, 0],
                Z[idx, 1],
                s=35,
                alpha=0.45,
                color=color,
                label=f"Cluster {c}"
            )

            # 95% confidence ellipse
            add_confidence_ellipse(
                ax=ax_pca,
                x=Z[idx, 0],
                y=Z[idx, 1],
                color=color
            )

            # Cluster centroid
            centroid = Z[idx].mean(axis=0)

            ax_pca.scatter(
                centroid[0],
                centroid[1],
                s=180,
                color=color,
                edgecolor="black",
                linewidth=1.2,
                marker="X",
                zorder=5
            )

        ax_pca.axhline(
            0.0,
            color="0.4",
            lw=0.8,
            alpha=0.5
        )

        ax_pca.axvline(
            0.0,
            color="0.4",
            lw=0.8,
            alpha=0.5
        )

        ax_pca.set_xlabel(
            f"PC1 ({explained[0]:.1f}% var.)"
        )

        ax_pca.set_ylabel(
            f"PC2 ({explained[1]:.1f}% var.)"
        )

        ax_pca.set_title(
            "D. Structural latent space"
        )

        ax_pca.grid(
            True,
            linestyle="--",
            alpha=0.25
        )

        ax_pca.legend(
            fontsize=8,
            loc="best",
            frameon=True
        )

    else:

        ax_pca.axis("off")

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------
    ax_shape.set_title("A. Normal-EF cluster mean shapes")
    ax_shape.set_aspect("equal")
    ax_shape.set_xlabel(r"$\beta_x(s)$")
    ax_shape.set_ylabel(r"$\beta_y(s)$")
    ax_shape.grid(True, linestyle="--", alpha=0.20)
    ax_shape.legend(fontsize=8, loc="best")

    ax_thick.set_title("B. Cluster mean thickness profiles")
    ax_thick.set_xlabel(r"Normalized contour parameter $s$")
    ax_thick.set_ylabel("Thickness")
    ax_thick.grid(True, linestyle="--", alpha=0.20)
    ax_thick.legend(fontsize=8, loc="best")

    ax_embed.set_title(r"C. Embedded cluster means $X(s)$")
    ax_embed.set_xlabel(r"$\beta_x$")
    ax_embed.set_ylabel(r"$\beta_y$")
    ax_embed.set_zlabel(r"$\alpha r(s)$")
    ax_embed.view_init(elev=28, azim=-62)
    ax_embed.legend(fontsize=8)

    try:
        ax_embed.set_box_aspect([1, 1, 0.7])
    except Exception:
        pass

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Structural heterogeneity among patients with normal EF. "
            "Panel A shows cluster-wise ventricular mean shapes, Panel B shows "
            "cluster-wise myocardial thickness profiles, Panel C shows the "
            "corresponding embedded shape-thickness representations, and Panel D "
            "shows cluster separation in structural feature space."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def plot_normal_ef_cluster_feature_space(features, labels, name):
    X = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    Z = pca.fit_transform(Xs)

    fig, ax = plt.subplots(figsize=(7, 6))

    for c in sorted(np.unique(labels)):
        idx = np.where(labels == c)[0]
        ax.scatter(
            Z[idx, 0],
            Z[idx, 1],
            s=45,
            alpha=0.85,
            label=f"Cluster {c}"
        )

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Normal EF structural clusters in feature space")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Two-dimensional PCA visualization of structural features among patients "
            "with normal EF, colored by cluster assignment."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_normal_ef_ef_boxplot(metadata_normal, labels, name):
    df = metadata_normal.copy()
    df["cluster"] = labels

    clusters = sorted(df["cluster"].unique())
    values = [
        df[df["cluster"] == c]["EF"].dropna().values
        for c in clusters
    ]

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.boxplot(values, labels=[f"C{c}" for c in clusters])
    ax.set_title("EF distribution within normal EF clusters")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("EF")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "EF distribution across structural clusters within the normal EF group. "
            "Similar EF distributions with distinct structures support latent heterogeneity."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_lambda_curve(lambda_results):
    df = lambda_results.copy()

    summary = (
        df.groupby("lambda")["srvf_geodesic_validation_mse"]
        .mean()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(
        summary["lambda"],
        summary["srvf_geodesic_validation_mse"],
        marker="o",
        lw=2
    )

    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel("CV SRVF geodesic MSE")
    ax.set_title("Lambda selection curve")

    plt.tight_layout()

    save_figure(
        fig,
        "fig_lambda_selection_curve",
        caption=(
            "Cross-validated SRVF Hilbert-sphere geodesic MSE as a function "
            "of the warping regularization parameter."
        ),
        label="fig:lambda_curve"
    )

    plt.close(fig)


from sklearn.metrics import roc_curve, auc


def plot_alignment_comparison(X_raw, X_aligned, name="fig_alignment_raw_vs_aligned", n=40):
    n = min(n, len(X_raw))

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    for i in range(n):
        axes[0].plot(X_raw[i, :, 0], X_raw[i, :, 1], alpha=0.35, lw=1)
        axes[1].plot(X_aligned[i, :, 0], X_aligned[i, :, 1], alpha=0.35, lw=1)

    axes[0].set_title("Before alignment")
    axes[1].set_title("After alignment")

    for ax in axes:
        ax.set_aspect("equal")
        ax.axis("off")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Representative ventricular contours before and after embedded SRVF alignment.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def srvf_distance_arrays_to_template(X_raw, X_aligned, X_template):
    q_template = normalize_srvf(srvf(X_template))

    before = []
    after = []

    for i in range(len(X_raw)):
        q0 = normalize_srvf(srvf(X_raw[i]))
        q1 = normalize_srvf(srvf(X_aligned[i]))

        before.append(srvf_sphere_geodesic_distance(q0, q_template))
        after.append(srvf_sphere_geodesic_distance(q1, q_template))

    return np.asarray(before), np.asarray(after)


def plot_srvf_distance_histogram(dist_before, dist_after, name="fig_srvf_geodesic_distance_histogram"):
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(dist_before, alpha=0.55, label="Before alignment")
    ax.hist(dist_after, alpha=0.55, label="After alignment")

    ax.set_xlabel("SRVF Hilbert-sphere geodesic distance")
    ax.set_ylabel("Frequency")
    ax.set_title("SRVF distance reduction after alignment")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Distribution of SRVF Hilbert-sphere geodesic distances to the training template before and after alignment.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_karcher_template(X_template, name="fig_karcher_template"):
    beta_template = X_template[:, :2]
    r_template = X_template[:, 2] / ALPHA_EMBED
    beta_out = reconstruct_outer(beta_template, r_template)

    fig, ax = plt.subplots(figsize=(6, 6))

    Bin = close_for_plot(beta_template)
    Bout = close_for_plot(beta_out)

    poly_x = np.concatenate([Bout[:, 0], Bin[::-1, 0]])
    poly_y = np.concatenate([Bout[:, 1], Bin[::-1, 1]])

    ax.fill(poly_x, poly_y, alpha=0.35, label="Mean myocardium")
    ax.plot(Bin[:, 0], Bin[:, 1], lw=3, label="Karcher mean endocardium")
    ax.plot(Bout[:, 0], Bout[:, 1], "--", lw=3, label="Mean epicardium")

    ax.set_title("Embedded SRVF Karcher template")
    ax.axis("equal")
    ax.axis("off")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Training-set embedded SRVF Karcher template reconstructed by inverse SRVF.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_ultrasound_overlay(image, beta_in, beta_out=None, name="fig_ultrasound_overlay"):
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.imshow(image, cmap="gray")
    ax.plot(beta_in[:, 0], beta_in[:, 1], lw=2.5, label="Endocardium")

    if beta_out is not None:
        ax.plot(beta_out[:, 0], beta_out[:, 1], "--", lw=2.5, label="Epicardium")

    ax.set_title("LV contours over ultrasound")
    ax.axis("off")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Example ultrasound frame with extracted left-ventricular contours.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def estimate_lv_landmarks(beta):
    """
    Approximate LV landmarks for visualization:
    apex = most leftward point;
    basal points = upper/lower points among rightmost contour region.
    """
    beta = np.asarray(beta)
    cent = beta.mean(axis=0)

    apex_idx = np.argmin(beta[:, 0])

    right_cut = np.percentile(beta[:, 0], 75)
    right_idx = np.where(beta[:, 0] >= right_cut)[0]

    if len(right_idx) < 2:
        right_idx = np.argsort(beta[:, 0])[-10:]

    upper_idx = right_idx[np.argmin(beta[right_idx, 1])]
    lower_idx = right_idx[np.argmax(beta[right_idx, 1])]

    return {
        "apex": beta[apex_idx],
        "basal_upper": beta[upper_idx],
        "basal_lower": beta[lower_idx]
    }


def normalize_for_display(beta):
    """
    Center and scale a contour for visual comparison.
    """
    b = beta.copy()
    b = b - b.mean(axis=0, keepdims=True)

    per = curve_perimeter(b)
    if per > 1e-8:
        b = b / per

    return b


def plot_clinical_to_srvf_pipeline_figure(
    image,
    beta_in,
    beta_out,
    r,
    beta_aligned,
    r_aligned,
    X_template,
    name="fig_clinical_to_srvf_pipeline"
):
    """
    Publication-style Figure 1:
    clinical image -> contours -> thickness -> embedding -> SRVF mean.
    """

    fig = plt.figure(figsize=(16, 10))

    ax1 = fig.add_subplot(2, 3, 1)
    ax2 = fig.add_subplot(2, 3, 2)
    ax3 = fig.add_subplot(2, 3, 3)
    ax4 = fig.add_subplot(2, 3, 4)
    ax5 = fig.add_subplot(2, 3, 5, projection="3d")
    ax6 = fig.add_subplot(2, 3, 6)

    endo_color = "#1f77b4"
    epi_color = "#d62728"
    mean_color = "black"

    # ========================================================
    # A. Ultrasound overlay with landmarks
    # ========================================================
    ax1.imshow(image, cmap="gray")

    B_in = close_for_plot(beta_in)
    B_out = close_for_plot(beta_out)

    ax1.plot(
        B_in[:, 0],
        B_in[:, 1],
        color=endo_color,
        lw=2.4,
        label="Endocardium"
    )

    ax1.plot(
        B_out[:, 0],
        B_out[:, 1],
        color=epi_color,
        linestyle="--",
        lw=2.4,
        label="Epicardium"
    )

    lm = estimate_lv_landmarks(beta_in)

    ax1.scatter(
        lm["apex"][0],
        lm["apex"][1],
        s=55,
        color="yellow",
        edgecolor="black",
        zorder=5,
        label="Apex"
    )

    ax1.scatter(
        [lm["basal_upper"][0], lm["basal_lower"][0]],
        [lm["basal_upper"][1], lm["basal_lower"][1]],
        s=45,
        color="white",
        edgecolor="black",
        zorder=5,
        label="Basal points"
    )

    ax1.set_title("A. ED ultrasound with LV contours")
    ax1.axis("off")
    ax1.legend(fontsize=8, loc="upper right")

    # ========================================================
    # B. Pose-normalized contours
    # ========================================================
    beta_in_norm = normalize_for_display(beta_in)
    beta_out_norm = normalize_for_display(beta_out)

    ax2.plot(
        *close_for_plot(beta_in_norm).T,
        color=endo_color,
        lw=2.5,
        label="Endocardium"
    )

    ax2.plot(
        *close_for_plot(beta_out_norm).T,
        color=epi_color,
        linestyle="--",
        lw=2.5,
        label="Epicardium"
    )

    ax2.set_title("B. Pose-normalized contours")
    ax2.set_aspect("equal")
    ax2.axis("off")
    ax2.legend(fontsize=8, loc="upper right")

    # ========================================================
    # C. Thickness spokes between contours
    # ========================================================
    ax3.imshow(image, cmap="gray")

    ax3.plot(
        B_in[:, 0],
        B_in[:, 1],
        color=endo_color,
        lw=2.2
    )

    ax3.plot(
        B_out[:, 0],
        B_out[:, 1],
        color=epi_color,
        linestyle="--",
        lw=2.2
    )

    _, Nvec = unit_tangent_normal(beta_in)
    beta_out_recon = reconstruct_outer(beta_in, r)

    spoke_idx = np.arange(0, len(beta_in), max(1, len(beta_in) // 25))

    segments = []
    values = []

    for j in spoke_idx:
        segments.append([
            beta_in[j],
            beta_out_recon[j]
        ])
        values.append(r[j])

    lc = LineCollection(
        segments,
        cmap="viridis",
        linewidths=2.0
    )
    lc.set_array(np.asarray(values))
    ax3.add_collection(lc)

    cbar = fig.colorbar(lc, ax=ax3, fraction=0.046, pad=0.04)
    cbar.set_label("Thickness")

    ax3.set_title("C. Myocardial thickness field")
    ax3.axis("off")

    # ========================================================
    # D. Thickness profile r(s)
    # ========================================================
    ax4.plot(
        s,
        r,
        color=mean_color,
        lw=2.8
    )

    ax4.set_title(r"D. Thickness profile $r(s)$")
    ax4.set_xlabel(r"Normalized contour parameter $s$")
    ax4.set_ylabel("Thickness")
    ax4.grid(True, linestyle="--", alpha=0.25)

    # ============================================================
    # Panel E: joint shape-thickness embedding
    # ============================================================

    ax = ax5

    # ------------------------------------------------------------
    # Use SRVF Karcher mean representation
    # ------------------------------------------------------------
    beta_mean = enforce_curve_closure(
        X_template[:, :2]
    )

    r_mean = enforce_scalar_periodicity(
        X_template[:, 2] / ALPHA_EMBED
    )

    # ------------------------------------------------------------
    # Embedded coordinates
    # ------------------------------------------------------------
    Xx = beta_mean[:, 0]
    Xy = beta_mean[:, 1]
    Xz = ALPHA_EMBED * r_mean

    # contour parameter
    s_vis = np.linspace(
        0.0,
        1.0,
        len(Xx)
    )

    # ------------------------------------------------------------
    # Ground projection (planar shadow)
    # ------------------------------------------------------------
    ax.plot(
        Xx,
        Xy,
        zs=0,
        zdir="z",
        color="black",
        lw=3,
        alpha=0.85
    )

    # ------------------------------------------------------------
    # Gradient-colored embedded curve
    # ------------------------------------------------------------
    cmap = plt.cm.plasma

    for i in range(len(Xx) - 1):

        ax.plot(
            Xx[i:i+2],
            Xy[i:i+2],
            Xz[i:i+2],
            color=cmap(s_vis[i]),
            lw=3
        )

    # ------------------------------------------------------------
    # Starting point marker
    # ------------------------------------------------------------
    ax.scatter(
        Xx[0],
        Xy[0],
        Xz[0],
        s=80,
        c="black",
        edgecolors="white",
        linewidths=1.2,
        zorder=10
    )
    
    # ------------------------------------------------------------
    # Legend
    # ------------------------------------------------------------
    from matplotlib.lines import Line2D

    legend_elements = [

        Line2D(
            [0],
            [0],
            color="black",
            lw=2.4,
            label="Planar contour projection"
        ),

        Line2D(
            [0],
            [0],
            color=plt.cm.plasma(0.50),
            lw=3,
            label=r'Embedded curve $X(s)$'
        ),

        Line2D(
            [0],
            [0],
            marker='o',
            color='w',
            markerfacecolor='black',
            markeredgecolor='white',
            markersize=8,
            lw=0,
            label=r'Start point $(s=0)$'
        )
    ]

    ax.legend(
        handles=legend_elements,
        fontsize=7,
        loc="upper left",
        frameon=True
    )
    
    # ------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------
    ax.set_xlabel(
        r'$\beta_x$',
        fontsize=12
    )

    ax.set_ylabel(
        r'$\beta_y$',
        fontsize=12
    )

    ax.set_zlabel(
        r'$\alpha r(s)$',
        fontsize=12
    )

    # ------------------------------------------------------------
    # Title
    # ------------------------------------------------------------
    ax.set_title(
        r"E. Joint shape-thickness embedding",
        fontsize=13,
        pad=10
    )

    # ------------------------------------------------------------
    # Camera angle
    # ------------------------------------------------------------
    ax.view_init(
        elev=28,
        azim=-62
    )

    # ------------------------------------------------------------
    # Transparent panes
    # ------------------------------------------------------------
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    # ------------------------------------------------------------
    # Soft grid
    # ------------------------------------------------------------
    ax.grid(
        alpha=0.25
    )

    # ------------------------------------------------------------
    # Aspect ratio
    # ------------------------------------------------------------
    try:

        ax.set_box_aspect(
            [1, 1, 0.7]
        )

    except Exception:

        pass

    # ========================================================
    # F. SRVF Karcher mean reconstruction
    # ========================================================
    beta_mean = enforce_curve_closure(X_template[:, :2])
    r_mean = enforce_scalar_periodicity(X_template[:, 2] / ALPHA_EMBED)
    beta_out_mean = reconstruct_outer(beta_mean, r_mean)

    for i in range(len(beta_aligned)):
        B = close_for_plot(beta_aligned[i])
        ax6.plot(
            B[:, 0],
            B[:, 1],
            color="0.75",
            alpha=0.25,
            lw=0.8
        )

    B_mean = close_for_plot(beta_mean)
    B_out_mean = close_for_plot(beta_out_mean)

    poly_x = np.concatenate([B_out_mean[:, 0], B_mean[::-1, 0]])
    poly_y = np.concatenate([B_out_mean[:, 1], B_mean[::-1, 1]])

    ax6.fill(
        poly_x,
        poly_y,
        color="0.80",
        alpha=0.45,
        label="Mean myocardium"
    )

    ax6.plot(
        B_mean[:, 0],
        B_mean[:, 1],
        color=endo_color,
        lw=3.0,
        label=r"Karcher mean endocardium"
    )

    ax6.plot(
        B_out_mean[:, 0],
        B_out_mean[:, 1],
        color=epi_color,
        linestyle="--",
        lw=3.0,
        label=r"Mean epicardium"
    )

    ax6.set_title("F. SRVF Karcher mean reconstruction")
    ax6.set_aspect("equal")
    ax6.axis("off")
    ax6.legend(fontsize=8, loc="lower left")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Clinical-to-geometric pipeline for the CAMUS data. "
            "An end-diastolic ultrasound frame is converted into endocardial "
            "and epicardial contours, from which a myocardial thickness profile "
            "is estimated. The curve--thickness pair is embedded as "
            "$X(s)=(\\beta(s),\\alpha r(s))$, mapped into SRVF space, aligned, "
            "and summarized by the SRVF Karcher mean."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def compute_srvf_tangent_fpca(q_aligned, mu, n_modes=3):
    return _srvf_numerics.tangent_fpca(
        [normalize_srvf(q) for q in q_aligned], mu, log_map_sphere,
        n_modes=n_modes, endpoint_duplicated=False
    )


def plot_srvf_tangent_fpca_modes(
    q_mean_lambda,
    q_aligned,
    X_template,
    beta_aligned=None,
    name="fig_srvf_tangent_fpca_modes"
):
    payload = compute_srvf_tangent_fpca(
        q_aligned=q_aligned,
        mu=q_mean_lambda,
        n_modes=3
    )

    mu = payload["mu"]
    pcs = payload["pcs"]
    scores = payload["scores"]
    n_modes = payload["n_modes"]

    fig, axes = plt.subplots(1, n_modes, figsize=(5.0 * n_modes, 5.0))

    if n_modes == 1:
        axes = [axes]

    beta_mean = enforce_curve_closure(X_template[:, :2])

    for k in range(n_modes):
        sd = np.std(scores[:, k])
        tau = 2.0 * sd

        v_minus = (-tau * pcs[k]).reshape(mu.shape)
        v_plus = (tau * pcs[k]).reshape(mu.shape)

        q_minus = exp_map_sphere(v_minus, mu)
        q_plus = exp_map_sphere(v_plus, mu)

        X_minus = inverse_srvf(q_minus, x0=X_template[0].copy())
        X_plus = inverse_srvf(q_plus, x0=X_template[0].copy())

        # Restore mode curves to the same visual scale/location as the Karcher mean
        target_len = curve_perimeter(beta_mean)

        for Xmode in [X_minus, X_plus]:

            Xmode[:] = enforce_curve_closure(Xmode)

            mode_len = curve_perimeter(Xmode[:, :2])
            scale = target_len / (mode_len + 1e-8)

            mode_cent = Xmode[:, :2].mean(axis=0, keepdims=True)
            mean_cent = beta_mean.mean(axis=0, keepdims=True)

            Xmode[:, :2] = mode_cent + scale * (Xmode[:, :2] - mode_cent)
            Xmode[:, :2] += mean_cent - Xmode[:, :2].mean(axis=0, keepdims=True)

            Xmode[:] = enforce_curve_closure(Xmode)

        ax = axes[k]

        if beta_aligned is not None:
            for beta in beta_aligned:
                B = close_for_plot(beta)
                ax.plot(
                    B[:, 0],
                    B[:, 1],
                    color="0.75",
                    alpha=0.25,
                    lw=0.8
                )

        B_minus = close_for_plot(X_minus[:, :2])
        B_plus = close_for_plot(X_plus[:, :2])
        B_mean = close_for_plot(beta_mean)                

        ax.plot(
            B_mean[:, 0],
            B_mean[:, 1],
            color="black",
            lw=3.0,
            label="SRVF Karcher mean"
        )

        ax.plot(
            B_minus[:, 0],
            B_minus[:, 1],
            color="#1f77b4",
            linestyle="--",
            lw=2.5,
            label=r"$-2$ SD"
        )

        ax.plot(
            B_plus[:, 0],
            B_plus[:, 1],
            color="#ff7f0e",
            linestyle="--",
            lw=2.5,
            label=r"$+2$ SD"
        )

        ax.set_title(f"SRVF tangent FPCA mode {k + 1}")
        ax.set_aspect("equal")
        ax.axis("off")

        if k == 0:
            ax.legend(fontsize=9, loc="upper right")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "SRVF tangent-space FPCA modes computed at the embedded SRVF "
            "Karcher mean. Background curves show aligned ventricular contours."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)

def plot_srvf_tangent_fpca_shape_thickness_modes(
    q_mean_lambda, q_aligned, X_template, beta_aligned, r_aligned,
    name="fig_srvf_tangent_fpca_shape_thickness_modes"
):
    """Both panels in a row are coordinates of one joint SRVF mode."""
    reference = np.array([build_embedding(b, r)
                          for b, r in zip(beta_aligned, r_aligned)])
    joint = _srvf_numerics.joint_mode_curves(
        q_mean_lambda, q_aligned, reference, compute_srvf_tangent_fpca,
        exp_map_sphere, ALPHA_EMBED, n_modes=3
    )
    n_modes = len(joint['modes'])
    fig, axes = plt.subplots(n_modes, 2, figsize=(12, 4.2*n_modes), squeeze=False)
    for k, (minus, plus) in enumerate(joint['modes']):
        for beta in beta_aligned:
            axes[k, 0].plot(*close_for_plot(beta).T, color='0.82', alpha=.22, lw=.8)
        for r in r_aligned:
            axes[k, 1].plot(s, r, color='0.82', alpha=.22, lw=.8)
        for curve, color, style, label in [
            (joint['mean'], 'black', '-', 'Reconstructed sphere mean'),
            (minus, '#1f77b4', '--', '-2 SD'),
            (plus, '#ff7f0e', '--', '+2 SD'),
        ]:
            axes[k, 0].plot(*close_for_plot(curve[:, :2]).T,
                            color=color, linestyle=style, lw=2.5, label=label)
            axes[k, 1].plot(s, curve[:, 2]/ALPHA_EMBED,
                            color=color, linestyle=style, lw=2.5, label=label)
        axes[k, 0].set_title(f'Joint mode {k+1}: shape')
        axes[k, 0].set_aspect('equal')
        axes[k, 0].axis('off')
        axes[k, 1].set_title(f'Joint mode {k+1}: thickness proxy')
        axes[k, 1].set_ylabel('r(s), pixels')
        axes[k, 1].grid(True, linestyle='--', alpha=.2)
        if k == 0:
            axes[k, 1].legend(fontsize=9)
    axes[-1, 1].set_xlabel('Normalized contour parameter')
    plt.tight_layout()
    save_figure(fig, name, caption=(
        'Shape and thickness-proxy coordinates reconstructed from the same joint '
        'SRVF tangent direction in each row. The ambient-sphere mean and modes '
        'undergo an explicit closure correction for display. A common isotropic '
        'scale and translation are used for all displayed curves.'), label=f'fig:{name}')
    plt.close(fig)
    diagnostics = [{key: value.tolist() if isinstance(value, np.ndarray) else value
                    for key, value in row.items()} for row in joint['diagnostics']]
    save_json({'geometry': 'ambient_sphere_approximation',
               'display_scale': joint['scale'], 'diagnostics': diagnostics},
              name + '_closure_diagnostics')
    payload = joint['fpca']
    # Preserve the original sample-variance denominator and amplitude convention.
    eigenvalues = payload['all_eigenvalues']
    table = pd.DataFrame({
        'mode': np.arange(1, n_modes+1), 'eigenvalue': eigenvalues[:n_modes],
        'explained_variance_percent': 100*eigenvalues[:n_modes]/eigenvalues.sum(),
        'cumulative_variance_percent': 100*np.cumsum(eigenvalues[:n_modes])/eigenvalues.sum()
    })
    save_latex_table(table, 'srvf_tangent_fpca_variance_explained',
                     'L2 variances of joint tangent modes; '
                     'variance ratios are unchanged by the common grid weight.',
                     'tab:srvf_tangent_fpca_variance_explained')
    
def plot_regression_results(y_true, y_pred, outcome="EF", model_name="model"):
    safe_model = model_name.replace(" ", "_")
    name = f"fig_prediction_{outcome}_{safe_model}"

    fig, ax = plt.subplots(figsize=(5, 5))

    ax.scatter(y_true, y_pred, alpha=0.65)

    lo = min(np.min(y_true), np.min(y_pred))
    hi = max(np.max(y_true), np.max(y_pred))

    ax.plot([lo, hi], [lo, hi], "k--", lw=2)

    ax.set_xlabel(f"Observed {outcome}")
    ax.set_ylabel(f"Predicted {outcome}")
    ax.set_title(f"{outcome} prediction: {model_name}")
    ax.grid(True, linestyle="--", alpha=0.25)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=f"Observed versus predicted {outcome} values on the test set using {model_name}.",
        label=f"fig:{name}"
    )

    plt.close(fig)

def plot_roc_all_models(classification_predictions, name="fig_low_EF_ROC_all_models"):
    """
    classification_predictions: list of dicts with keys:
        model, y_true, y_prob
    """
    fig, ax = plt.subplots(figsize=(6, 5))

    for item in classification_predictions:
        y_true = np.asarray(item["y_true"])
        y_prob = np.asarray(item["y_prob"])
        model_name = item["model"]

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_value = auc(fpr, tpr)

        ax.plot(
            fpr,
            tpr,
            lw=2.2,
            label=f"{model_name} (AUC={auc_value:.3f})"
        )

    ax.plot([0, 1], [0, 1], "k--", lw=1.5)

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Low-EF classification: ROC curves")
    ax.grid(True, linestyle="--", alpha=0.25)
    ax.legend(fontsize=8)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="ROC curves for all low-EF classification models on the test set.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_regression_and_roc_side_by_side(
    regression_predictions,
    classification_predictions,
    outcome="EF",
    name="fig_prediction_and_roc_combined"
):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # ========================================================
    # Left: regression
    # ========================================================
    ax = axes[0]

    all_true = []
    all_pred = []

    for item in regression_predictions:

        model_name = item["model"]

        y_true_plot, y_pred_plot = _clean_regression_predictions_for_plot(
            item["y_true"],
            item["y_pred"]
        )

        if len(y_true_plot) < 2:
            continue

        all_true.extend(y_true_plot)
        all_pred.extend(y_pred_plot)

        r2_val = r2_score(
            y_true_plot,
            y_pred_plot
        )

        ax.scatter(
            y_true_plot,
            y_pred_plot,
            alpha=0.35,
            s=24
        )

        if len(np.unique(y_true_plot)) >= 2:

            coef = np.polyfit(
                y_true_plot,
                y_pred_plot,
                deg=1
            )

            x_line = np.linspace(
                np.min(y_true_plot),
                np.max(y_true_plot),
                100
            )

            y_line = coef[0] * x_line + coef[1]

            ax.plot(
                x_line,
                y_line,
                lw=2.2,
                label=f"{model_name} ($R^2$={r2_val:.3f})"
            )

    if len(all_true) > 0 and len(all_pred) > 0:

        all_true = np.asarray(all_true)
        all_pred = np.asarray(all_pred)

        lo = min(
            np.min(all_true),
            np.min(all_pred)
        )

        hi = max(
            np.max(all_true),
            np.max(all_pred)
        )

        pad = 0.05 * (hi - lo + 1e-8)

        lo = max(0.0, lo - pad)
        hi = min(100.0, hi + pad)

        ax.plot(
            [lo, hi],
            [lo, hi],
            "k--",
            lw=2.2,
            label="Ideal prediction"
        )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("Observed EF (%)")
    ax.set_ylabel("Predicted EF (%)")
    ax.set_title("Observed versus predicted EF")
    ax.grid(True, linestyle="--", alpha=0.25)
    ax.legend(fontsize=7, loc="best")

    
    # ========================================================
    # Right: ROC
    # ========================================================
    ax = axes[1]

    for item in classification_predictions:
        y_true = np.asarray(item["y_true"])
        y_prob = np.asarray(item["y_prob"])
        model_name = item["model"]

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_value = auc(fpr, tpr)

        ax.plot(
            fpr,
            tpr,
            lw=2.2,
            label=f"{model_name} (AUC={auc_value:.3f})"
        )

    ax.plot([0, 1], [0, 1], "k--", lw=1.5)

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Low-EF ROC")
    ax.grid(True, linestyle="--", alpha=0.25)
    ax.legend(fontsize=7)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Combined prediction and classification performance. Left: observed "
            "versus predicted EF values for all regression models. Right: ROC "
            "curves for all low-EF classification models."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def _clean_regression_predictions_for_plot(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true >= 0)
        & (y_true <= 100)
        & (y_pred >= 0)
        & (y_pred <= 100)
    )

    return y_true[valid], y_pred[valid]


def plot_regression_all_models(
    regression_predictions,
    outcome="EF",
    name=None
):
    """
    Observed versus predicted outcome for all regression models.
    Uses equal x/y limits so the ideal prediction line is a true 45-degree line.
    """

    if name is None:
        name = f"fig_prediction_{outcome}_all_models"

    fig, ax = plt.subplots(figsize=(7, 6))

    all_true = []
    all_pred = []

    for item in regression_predictions:

        model_name = item["model"]

        y_true_plot, y_pred_plot = _clean_regression_predictions_for_plot(
            item["y_true"],
            item["y_pred"]
        )

        if len(y_true_plot) < 2:
            continue

        all_true.extend(y_true_plot)
        all_pred.extend(y_pred_plot)

        r2_val = r2_score(
            y_true_plot,
            y_pred_plot
        )

        ax.scatter(
            y_true_plot,
            y_pred_plot,
            alpha=0.45,
            s=32,
            label=f"{model_name} ($R^2$={r2_val:.3f})"
        )

    if len(all_true) == 0:
        plt.close(fig)
        return

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    lo = min(
        np.min(all_true),
        np.min(all_pred)
    )

    hi = max(
        np.max(all_true),
        np.max(all_pred)
    )

    pad = 0.05 * (hi - lo + 1e-8)

    lo = max(0.0, lo - pad)
    hi = min(100.0, hi + pad)

    ax.plot(
        [lo, hi],
        [lo, hi],
        "k--",
        lw=2.5,
        label="Ideal prediction"
    )

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(f"Observed {outcome} (%)")
    ax.set_ylabel(f"Predicted {outcome} (%)")
    ax.set_title(f"Observed versus predicted {outcome}")
    ax.grid(True, linestyle="--", alpha=0.25)
    ax.legend(fontsize=7, loc="best")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            f"Observed versus predicted {outcome} values for all regression "
            "models on the independent test set. The dashed line represents "
            "ideal prediction."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def plot_roc_curve_result(y_true, y_prob, model_name="model"):
    safe_model = model_name.replace(" ", "_")
    name = f"fig_low_EF_ROC_{safe_model}"

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(5, 5))

    ax.plot(fpr, tpr, lw=2.5, label=f"AUC = {auc_value:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1.5)

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"Low-EF ROC: {model_name}")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.25)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=f"ROC curve for low-EF classification on the test set using {model_name}.",
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_train_test_ef_distribution(meta_train, meta_test, name="fig_train_test_EF_distribution"):
    if "EF" not in meta_train.columns or "EF" not in meta_test.columns:
        return

    ef_train = meta_train["EF"].dropna()
    ef_test = meta_test["EF"].dropna()

    if len(ef_train) == 0 or len(ef_test) == 0:
        return

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(ef_train, alpha=0.55, label="Train")
    ax.hist(ef_test, alpha=0.55, label="Test")

    ax.axvline(EF_CLASSIFICATION_THRESHOLD, linestyle="--", lw=2, label="Low-EF threshold")

    ax.set_xlabel("EF")
    ax.set_ylabel("Frequency")
    ax.set_title("EF distribution in train and test sets")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Distribution of EF values in the training and test sets.",
        label=f"fig:{name}"
    )

    plt.close(fig)
    
def plot_feature_clusters(features, labels, name="fig_structural_cluster_feature_space"):
    X = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    Z = pca.fit_transform(Xs)

    fig, ax = plt.subplots(figsize=(6, 5))

    scatter = ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap="coolwarm", alpha=0.75)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Structural clusters in feature space")
    ax.grid(True, linestyle="--", alpha=0.25)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption="Two-dimensional PCA visualization of structural feature clusters.",
        label=f"fig:{name}"
    )

    plt.close(fig)

def save_clinical_cohort_summary(metadata, beta_raw, r_raw):
    df = metadata.copy()

    df["area"] = [curve_area(b) for b in beta_raw]
    df["perimeter"] = [curve_perimeter(b) for b in beta_raw]
    df["compactness"] = [compactness(b) for b in beta_raw]
    df["mean_thickness"] = np.mean(r_raw, axis=1)
    df["sd_thickness"] = np.std(r_raw, axis=1)

    if "EF" in df.columns:
        df["EF_group"] = np.where(
            df["EF"] < EF_CLASSIFICATION_THRESHOLD,
            "Low EF",
            "Normal EF"
        )
    else:
        df["EF_group"] = "Overall"

    variables = [
        "EF", "EDV", "ESV",
        "area", "perimeter", "compactness",
        "mean_thickness", "sd_thickness"
    ]

    rows = []

    for var in variables:
        if var not in df.columns:
            continue

        overall = df[var].dropna()
        normal = df.loc[df["EF_group"] == "Normal EF", var].dropna()
        low = df.loc[df["EF_group"] == "Low EF", var].dropna()

        if len(normal) >= 2 and len(low) >= 2:
            try:
                _, pval = mannwhitneyu(normal, low, alternative="two-sided")
            except Exception:
                pval = np.nan
        else:
            pval = np.nan

        rows.append({
            "variable": var,
            "overall_mean": overall.mean(),
            "overall_sd": overall.std(),
            "normal_EF_mean": normal.mean(),
            "normal_EF_sd": normal.std(),
            "low_EF_mean": low.mean(),
            "low_EF_sd": low.std(),
            "p_mannwhitney": pval
        })

    out = pd.DataFrame(rows)

    save_latex_table(
        out,
        "clinical_cohort_summary",
        "Clinical and geometric characteristics of the CAMUS cohort.",
        "tab:clinical_cohort_summary"
    )

    return out


def save_lambda_selection_stability(lambda_results):
    df = lambda_results.copy()

    out = (
        df.groupby("lambda")
        .agg(
            selection_frequency=("best_lambda", lambda x: np.sum(x == x.name)),
            mean_validation_mse=("srvf_geodesic_validation_mse", "mean"),
            sd_validation_mse=("srvf_geodesic_validation_mse", "std")
        )
        .reset_index()
    )

    save_latex_table(
        out,
        "lambda_selection_stability",
        "Stability of warping regularization parameter selection.",
        "tab:lambda_selection_stability"
    )

    return out


def save_clustering_model_selection_table(clustering_results):
    if len(clustering_results) == 0:
        return pd.DataFrame()

    df = clustering_results.copy()

    out = (
        df.groupby(["model", "k"])
        .agg(
            mean_cv_silhouette=("cv_silhouette", "mean"),
            sd_cv_silhouette=("cv_silhouette", "std"),
            n_evaluations=("cv_silhouette", "count")
        )
        .reset_index()
        .sort_values("mean_cv_silhouette", ascending=False)
    )

    save_latex_table(
        out,
        "clustering_model_selection_summary",
        "Cross-validated clustering model selection summary.",
        "tab:clustering_model_selection_summary"
    )

    return out


def save_simplified_normal_ef_cluster_table(normal_cluster_comparison_results):
    if len(normal_cluster_comparison_results) == 0:
        return pd.DataFrame()

    keep_metrics = [
        "EF",
        "EDV",
        "ESV",
        "area",
        "perimeter",
        "compactness",
        "mean_thickness",
        "sd_thickness",
        "shape_distance_to_template",
        "srvf_distance_to_template"
    ]

    df = normal_cluster_comparison_results.copy()
    df = df[df["metric"].isin(keep_metrics)]

    if len(df) == 0:
        return pd.DataFrame()

    out = (
        df.groupby("metric")
        .agg(
            cluster_1_mean=("cluster_1_mean", "mean"),
            cluster_2_mean=("cluster_2_mean", "mean"),
            mean_difference=("mean_difference", "mean"),
            cohen_d=("cohen_d", "mean"),
            p_mannwhitney=("p_mannwhitney", "mean")
        )
        .reset_index()
    )

    save_latex_table(
        out,
        "normal_EF_cluster_comparison_simplified",
        "Simplified comparison of structural clusters among patients with normal EF.",
        "tab:normal_EF_cluster_comparison_simplified"
    )

    return out


def save_best_model_hyperparameter_tables(regression_results, classification_results):
    if len(regression_results) > 0:
        reg_best = (
            regression_results
            .sort_values(["outcome", "rmse"])
            .groupby("outcome")
            .head(1)
            .reset_index(drop=True)
        )

        reg_best = reg_best[
            ["outcome", "model", "mae", "rmse", "r2", "corr", "cv_rmse", "best_params"]
        ]

        save_latex_table(
            reg_best,
            "regression_best_model_hyperparameters",
            "Best regression models and selected hyperparameters.",
            "tab:regression_best_model_hyperparameters"
        )

    if len(classification_results) > 0:
        cls_best = (
            classification_results
            .sort_values(["outcome", "auc"], ascending=[True, False])
            .groupby("outcome")
            .head(1)
            .reset_index(drop=True)
        )

        cls_best = cls_best[
            ["outcome", "model", "accuracy", "balanced_accuracy",
             "f1", "auc", "sensitivity", "specificity",
             "cv_auc_or_balacc", "best_params"]
        ]

        save_latex_table(
            cls_best,
            "classification_best_model_hyperparameters",
            "Best classification models and selected hyperparameters.",
            "tab:classification_best_model_hyperparameters"
        )
        
# ============================================================
# 15. Result aggregation
# ============================================================

def summarize_results(df, group_cols, metric_cols):
    rows = []

    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        for m in metric_cols:
            row[f"{m}_mean"] = g[m].mean()
            row[f"{m}_sd"] = g[m].std()

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 16. Repeated train/test analysis
# ============================================================

def make_split_indices(metadata, repeat_id):
    n = len(metadata)
    idx = np.arange(n)

    stratify = None
    y = None

    # --------------------------------------------------------
    # Stratification based on EF threshold
    # --------------------------------------------------------
    if "EF" in metadata.columns and metadata["EF"].notna().sum() == n:
        y = (metadata["EF"].values < EF_CLASSIFICATION_THRESHOLD).astype(int)

        if len(np.unique(y)) == 2 and min(np.bincount(y)) >= 3:
            stratify = y

    # --------------------------------------------------------
    # Train / test split
    # --------------------------------------------------------
    train_idx, test_idx = train_test_split(
        idx,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE + repeat_id,
        shuffle=True,
        stratify=stratify
    )

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------
    print(
        f"  Split sizes: train={len(train_idx)}, test={len(test_idx)} "
        f"(test_size={TEST_SIZE:.2f})",
        flush=True
    )

    # EF class balance
    if y is not None:
        y_train = y[train_idx]
        y_test = y[test_idx]

        print(
            f"  Low-EF counts: "
            f"train={y_train.sum()}/{len(y_train)}, "
            f"test={y_test.sum()}/{len(y_test)}",
            flush=True
        )

    # EF stats
    if "EF" in metadata.columns and metadata["EF"].notna().sum() > 0:
        ef_train = metadata.iloc[train_idx]["EF"].dropna()
        ef_test  = metadata.iloc[test_idx]["EF"].dropna()

        print(
            f"  EF stats (train): "
            f"mean={ef_train.mean():.2f}, std={ef_train.std():.2f}",
            flush=True
        )

        print(
            f"  EF stats (test): "
            f"mean={ef_test.mean():.2f}, std={ef_test.std():.2f}",
            flush=True
        )

    return train_idx, test_idx


def evaluate_alignment_split_geodesic(beta_raw, r_raw, X_raw,
                                      beta_a, r_a, X_a,
                                      X_template, q_reference=None, q_aligned=None):
    """
    Alignment quality is measured using SRVF Hilbert-sphere
    geodesic distance to the training template.
    """

    q_template = normalize_srvf(srvf(X_template)) if q_reference is None else normalize_srvf(q_reference)

    srvf_before = []
    srvf_after = []
    gamma_placeholder = []

    for i in range(len(X_raw)):
        q_raw_i = normalize_srvf(srvf(X_raw[i]))
        q_a_i = normalize_srvf(srvf(X_a[i])) if q_aligned is None else normalize_srvf(q_aligned[i])

        d_before = srvf_sphere_geodesic_distance(q_raw_i, q_template)
        d_after = srvf_sphere_geodesic_distance(q_a_i, q_template)

        srvf_before.append(d_before)
        srvf_after.append(d_after)

    srvf_before = np.asarray(srvf_before)
    srvf_after = np.asarray(srvf_after)

    return {
        "srvf_geodesic_before_mean": np.mean(srvf_before),
        "srvf_geodesic_before_sd": np.std(srvf_before),
        "srvf_geodesic_after_mean": np.mean(srvf_after),
        "srvf_geodesic_after_sd": np.std(srvf_after),
        "srvf_geodesic_change_percent": (
            100.0 * (np.mean(srvf_before) - np.mean(srvf_after))
            / max(np.mean(srvf_before), 1e-12)
        )
    }


def restore_feature_template(X_template, X_train_a):
    """Reuse the v01 training-only curve-domain restoration unchanged."""
    X_template = X_template.copy()
    X_template = center_embedding(X_template)
    X_template = enforce_curve_closure(X_template)

    # Restore XY scale
    mean_xy_length_data = np.mean([
        curve_perimeter(X_train_a[i][:, :2])
        for i in range(len(X_train_a))
    ])

    template_xy_length = curve_perimeter(X_template[:, :2])
    xy_scale = mean_xy_length_data / (template_xy_length + 1e-8)

    xy_cent = X_template[:, :2].mean(axis=0, keepdims=True)
    X_template[:, :2] = xy_cent + xy_scale * (X_template[:, :2] - xy_cent)

    # Restore XY location
    target_xy_cent = X_train_a[:, :, :2].reshape(-1, 2).mean(axis=0, keepdims=True)
    current_xy_cent = X_template[:, :2].mean(axis=0, keepdims=True)
    X_template[:, :2] += target_xy_cent - current_xy_cent

    # Restore the scalar level for legacy curve-domain displays only.
    # Direct SRVF alignment and primary distances use q_mean_lambda.
    X_template[:, 2] += X_train_a[:, :, 2].mean() - X_template[:, 2].mean()

    # Enforce scalar periodicity in the embedded thickness coordinate
    X_template[:, 2] = enforce_scalar_periodicity(X_template[:, 2])
    return X_template


def fit_inner_representation(beta_train, r_train, X_train,
                             beta_validation, r_validation, X_validation,
                             repeat_id=0):
    """Fit lambda, alignment and PCA using this inner training subset only."""
    best_lambda, _, _ = select_lambda_5fold_cv_real(
        beta_train, r_train, X_train, repeat_id=repeat_id
    )
    fitted = run_alignment(beta_train, r_train, X_train, lambda_warp=best_lambda)
    beta_train_a, r_train_a, X_train_a = fitted[3:6]
    q_reference, X_alignment_template = fitted[-2:]
    aligned = align_to_fixed_template(
        beta_validation, r_validation, X_validation,
        X_template=X_alignment_template, lambda_warp=best_lambda,
        q_reference=q_reference, return_srvfs=True
    )
    X_feature_template = restore_feature_template(X_alignment_template, X_train_a)
    training, validation = build_features_train_test(
        beta_train_a, r_train_a, X_train_a,
        aligned[3], aligned[4], aligned[5], X_feature_template
    )
    return training, validation, {
        'best_lambda': float(best_lambda),
        'n_lambda_training_rows': len(X_train),
        'mean_closure_norm': _srvf_numerics.closure_diagnostics(q_reference)['closure_norm'],
        'feature_family': 'unchanged_mixed_scalar_and_coordinate_PCA',
        'lambda_selection': 'existing_CV_on_inner_training_rows_only'
    }


def prepare_inner_cv_features(beta, r, X, row_ids, repeat_id=0, n_folds=N_FOLDS):
    def fit(training_beta, training_r, training_X, validation_beta, validation_r, validation_X):
        return fit_inner_representation(
            training_beta, training_r, training_X,
            validation_beta, validation_r, validation_X, repeat_id=repeat_id
        )
    return _camus_cv.prepare_folds(
        beta, r, X, row_ids, fit, n_folds=n_folds, random_state=RANDOM_STATE
    )


def run_one_repeat(
    repeat_id,
    beta_in_raw,
    beta_out_raw,
    r_raw,
    X_raw,
    metadata
):
    print("==========================================")
    print(f"Repeat {repeat_id + 1}/{N_REPEATS}")
    print("==========================================")

    # --------------------------------------------------------
    # 1. Train / test split
    # --------------------------------------------------------
    train_idx, test_idx = make_split_indices(metadata, repeat_id)

    beta_train = beta_in_raw[train_idx]
    r_train = r_raw[train_idx]
    X_train = X_raw[train_idx]

    beta_test = beta_in_raw[test_idx]
    r_test = r_raw[test_idx]
    X_test = X_raw[test_idx]

    meta_train = metadata.iloc[train_idx].reset_index(drop=True)
    meta_test = metadata.iloc[test_idx].reset_index(drop=True)

    # --------------------------------------------------------
    # 2. Lambda selection by K-fold CV within training set only
    # --------------------------------------------------------
    print(f"  Selecting lambda by {N_FOLDS}-fold cross-validation on training set")

    best_lambda, lambda_table, lambda_summary = select_lambda_5fold_cv_real(
        beta_train,
        r_train,
        X_train,
        repeat_id=repeat_id
    )

    # --------------------------------------------------------
    # 3. Refit alignment on full training set
    # --------------------------------------------------------
    (
        beta_train_w,
        r_train_w,
        X_train_w,
        beta_train_a,
        r_train_a,
        X_train_a,
        gam_train,
        q_train_a,
        q_mean_lambda,
        X_template
    ) = run_alignment(
        beta_train,
        r_train,
        X_train,
        lambda_warp=best_lambda
    )


    q_train_aligned = q_train_a
    X_alignment_template = X_template.copy()

    # --------------------------------------------------------
    # Geometric normalization (translation only)
    # --------------------------------------------------------
    X_template = restore_feature_template(X_template, X_train_a)

    # 4. Align test set to fixed training template
    # --------------------------------------------------------
    (
        beta_test_w,
        r_test_w,
        X_test_w,
        beta_test_a,
        r_test_a,
        X_test_a,
        gam_test, q_test_a
    ) = align_to_fixed_template(
        beta_test,
        r_test,
        X_test,
        X_template=X_alignment_template,
        lambda_warp=best_lambda, q_reference=q_mean_lambda, return_srvfs=True
    )

    # --------------------------------------------------------
    # 5. Alignment evaluation using SRVF geodesic distance
    # --------------------------------------------------------
    align_train = evaluate_alignment_split_geodesic(
        beta_train,
        r_train,
        X_train,
        beta_train_a,
        r_train_a,
        X_train_a,
        X_template, q_reference=q_mean_lambda, q_aligned=q_train_a
    )

    align_test = evaluate_alignment_split_geodesic(
        beta_test,
        r_test,
        X_test,
        beta_test_a,
        r_test_a,
        X_test_a,
        X_template, q_reference=q_mean_lambda, q_aligned=q_test_a
    )

    save_json({
        'geometry': 'ambient_sphere_approximation',
        'mean': {k: v.tolist() if isinstance(v, np.ndarray) else v
                 for k, v in _srvf_numerics.closure_diagnostics(q_mean_lambda).items()},
        'train_closure_norms': [_srvf_numerics.closure_diagnostics(q)['closure_norm'] for q in q_train_a],
        'test_closure_norms': [_srvf_numerics.closure_diagnostics(q)['closure_norm'] for q in q_test_a]
    }, f'closure_diagnostics_repeat_{repeat_id:03d}')
    alignment_df = pd.DataFrame([
        {
            "repeat": repeat_id,
            "split": "train",
            "best_lambda": best_lambda,
            **align_train
        },
        {
            "repeat": repeat_id,
            "split": "test",
            "best_lambda": best_lambda,
            **align_test
        }
    ])

    # --------------------------------------------------------
    # 6. Feature extraction: train only fit, test projection
    # --------------------------------------------------------
    Xfeat_train, Xfeat_test = build_features_train_test(
        beta_train_a,
        r_train_a,
        X_train_a,
        beta_test_a,
        r_test_a,
        X_test_a,
        X_template
    )

    if repeat_id == 0:
        plot_alignment_comparison(
            X_raw=np.concatenate([X_train, X_test], axis=0),
            X_aligned=np.concatenate([X_train_a, X_test_a], axis=0),
            name="fig_alignment_raw_vs_aligned"
        )

        dist_before, dist_after = srvf_distance_arrays_to_template(
            X_raw=np.concatenate([X_train, X_test], axis=0),
            X_aligned=np.concatenate([X_train_a, X_test_a], axis=0),
            X_template=X_template
        )

        plot_srvf_distance_histogram(
            dist_before,
            dist_after,
            name="fig_srvf_geodesic_distance_histogram"
        )

        plot_karcher_template(
            X_template,
            name="fig_karcher_template"
        )

        plot_train_test_ef_distribution(
            meta_train,
            meta_test,
            name="fig_train_test_EF_distribution"
        )

        plot_srvf_tangent_fpca_modes(
            q_mean_lambda=q_mean_lambda,
            q_aligned=q_train_aligned,
            X_template=X_template,
            beta_aligned=beta_train_a,
            name="fig_srvf_tangent_fpca_modes"
        )

        plot_srvf_tangent_fpca_shape_thickness_modes(
            q_mean_lambda=q_mean_lambda,
            q_aligned=q_train_aligned,
            X_template=X_template,
            beta_aligned=beta_train_a,
            r_aligned=r_train_a,
            name="fig_srvf_tangent_fpca_shape_thickness_modes"
        )

        # ----------------------------------------------------
        # Ultrasound overlay + clinical-to-SRVF pipeline
        # ----------------------------------------------------
        try:

            ex = 0

            image_path = meta_train.iloc[ex]["image_file"]

            if image_path is not None and isinstance(image_path, str):

                img = load_medical_image(image_path)

                # --------------------------------------------
                # Raw ultrasound overlay
                # --------------------------------------------
                plot_ultrasound_overlay(
                    image=img,
                    beta_in=beta_train[ex],
                    beta_out=beta_out_raw[train_idx][ex],
                    name="fig_ultrasound_overlay_example"
                )

                # --------------------------------------------
                # Full clinical-to-SRVF pipeline figure
                # --------------------------------------------
                plot_clinical_to_srvf_pipeline_figure(
                    image=img,
                    beta_in=beta_train[ex],
                    beta_out=beta_out_raw[train_idx][ex],
                    r=r_train[ex],
                    beta_aligned=beta_train_a,
                    r_aligned=r_train_a,
                    X_template=X_template,
                    name="fig_clinical_to_srvf_pipeline"
                )

        except Exception as e:

            print(
                f"    [WARN] ultrasound/SRVF visualization skipped: {e}",
                flush=True
            )


    # --------------------------------------------------------
    # Cache fold-fitted representations, never external validation/test data.
    inner_feature_cache = {}
    def inner_features(valid=None):
        if valid is None:
            valid = np.ones(len(X_train), dtype=bool)
        positions = np.flatnonzero(valid)
        key = tuple(int(i) for i in positions)
        if key not in inner_feature_cache:
            inner_feature_cache[key] = prepare_inner_cv_features(
                beta_train[positions], r_train[positions], X_train[positions],
                np.asarray(train_idx)[positions], repeat_id=repeat_id
            )
        return inner_feature_cache[key]

    # 7. Regression analysis
    # --------------------------------------------------------
    regression_rows = []
    regression_predictions_for_fig = []    

    for outcome in ["EF", "EDV", "ESV"]:
        if outcome not in metadata.columns:
            continue

        y_train = meta_train[outcome].values
        y_test = meta_test[outcome].values

        valid_train = np.isfinite(y_train)
        valid_test = np.isfinite(y_test)

        if valid_train.sum() < 10 or valid_test.sum() < 3:
            continue

        Xt = Xfeat_train.loc[valid_train]
        Xs = Xfeat_test.loc[valid_test]

        yt = y_train[valid_train]
        ys = y_test[valid_test]

        regression_folds = inner_features(valid_train)

        for model_name, spec in regression_model_specs().items():
            print(f"  Regression: {outcome}, {model_name}")

            try:
                best_model, best_params, cv_score = tune_regression_model_cv(
                    spec,
                    Xt,
                    yt, fold_features=regression_folds
                )

                pred_test = best_model.predict(Xs)

                if repeat_id == 0 and outcome == "EF":
                    regression_predictions_for_fig.append({
                        "model": model_name,
                        "y_true": ys,
                        "y_pred": pred_test
                    })
                                    
                metrics = evaluate_regression(pred_test, ys)

                if repeat_id == 0:
                    plot_regression_results(
                    y_true=ys,
                    y_pred=pred_test,
                    outcome=outcome,
                    model_name=model_name
                    )

                row = {
                    "repeat": repeat_id,
                    "outcome": outcome,
                    "model": model_name,
                    "cv_rmse": cv_score,
                    "best_params": json.dumps(best_params)
                }
                row.update(metrics)
                regression_rows.append(row)

            except Exception as e:
                print(f"    [WARN] skipped regression {outcome}-{model_name}: {e}")

    regression_df = pd.DataFrame(regression_rows)

    # --------------------------------------------------------
    # 8. Classification analysis: EF < threshold
    # --------------------------------------------------------
    classification_rows = []
    classification_predictions_for_fig = []

    if "EF" in metadata.columns:
        y_train_raw = meta_train["EF"].values
        y_test_raw = meta_test["EF"].values

        valid_train = np.isfinite(y_train_raw)
        valid_test = np.isfinite(y_test_raw)

        if valid_train.sum() >= 10 and valid_test.sum() >= 3:

            y_train = (
                y_train_raw[valid_train] < EF_CLASSIFICATION_THRESHOLD
            ).astype(int)

            y_test = (
                y_test_raw[valid_test] < EF_CLASSIFICATION_THRESHOLD
            ).astype(int)

            if len(np.unique(y_train)) == 2 and len(np.unique(y_test)) == 2:

                Xt = Xfeat_train.loc[valid_train]
                Xs = Xfeat_test.loc[valid_test]

                classification_folds = inner_features(valid_train)

                for model_name, spec in classification_model_specs().items():
                    print(f"  Classification: {model_name}")

                    try:
                        best_model, best_params, cv_score = tune_classification_model_cv(
                            spec,
                            Xt,
                            y_train, fold_features=classification_folds
                        )

                        metrics = evaluate_classification(
                            best_model,
                            Xs,
                            y_test
                        )

                        if repeat_id == 0 and hasattr(best_model, "predict_proba"):
                            y_prob = best_model.predict_proba(Xs)[:, 1]

                            classification_predictions_for_fig.append({
                                "model": model_name,
                                "y_true": y_test,
                                "y_prob": y_prob
                            })

                        row = {
                            "repeat": repeat_id,
                            "outcome": f"EF < {EF_CLASSIFICATION_THRESHOLD}",
                            "model": model_name,
                            "cv_auc_or_balacc": cv_score,
                            "best_params": json.dumps(best_params)
                        }
                        row.update(metrics)
                        classification_rows.append(row)

                    except Exception as e:
                        print(f"    [WARN] skipped classification {model_name}: {e}")

    classification_df = pd.DataFrame(classification_rows)

    # --------------------------------------------------------
    # Combined regression / classification figures
    # --------------------------------------------------------
    if repeat_id == 0 and len(regression_predictions_for_fig) > 0:
        plot_regression_all_models(
            regression_predictions_for_fig,
            outcome="EF",
            name="fig_prediction_EF_all_models"
        )

    if repeat_id == 0 and len(classification_predictions_for_fig) > 0:
        plot_roc_all_models(
            classification_predictions_for_fig,
            name="fig_low_EF_ROC_all_models"
        )

    if (
        repeat_id == 0
        and len(regression_predictions_for_fig) > 0
        and len(classification_predictions_for_fig) > 0
    ):
        plot_regression_and_roc_side_by_side(
            regression_predictions=regression_predictions_for_fig,
            classification_predictions=classification_predictions_for_fig,
            outcome="EF",
            name="fig_EF_prediction_and_low_EF_ROC_combined"
        )

    # --------------------------------------------------------
    # 9. Clustering: select model/k by train CV, evaluate test
    # --------------------------------------------------------
    clustering_rows = []
    cluster_summary_rows = []
    normal_cluster_rows = []
    normal_comparison_rows = []

    print("  Clustering analysis")

    try:
        scaler_clust = StandardScaler()
        Xc_train = scaler_clust.fit_transform(Xfeat_train)
        Xc_test = scaler_clust.transform(Xfeat_test)

        cluster_model, cluster_name, best_k, cluster_table = select_k_train_cv(
            Xc_train, fold_features=inner_features()
        )

        cluster_table["repeat"] = repeat_id
        cluster_table["selected_model"] = cluster_name
        cluster_table["selected_k"] = best_k

        clustering_rows.append(cluster_table)

        labels_train = assign_clusters(cluster_model, cluster_name, Xc_train)
        labels_test = assign_clusters(cluster_model, cluster_name, Xc_test)

        if repeat_id == 0:
            plot_feature_clusters(
                features=Xfeat_test,
                labels=labels_test,
                name="fig_structural_cluster_feature_space"
            )

        cluster_train = cluster_summary(meta_train, labels_train)
        cluster_train["repeat"] = repeat_id
        cluster_train["split"] = "train"
        cluster_train["model"] = cluster_name
        cluster_train["k"] = best_k

        cluster_test = cluster_summary(meta_test, labels_test)
        cluster_test["repeat"] = repeat_id
        cluster_test["split"] = "test"
        cluster_test["model"] = cluster_name
        cluster_test["k"] = best_k

        cluster_summary_rows.extend([cluster_train, cluster_test])

        normal_summary, normal_comparison, normal_payload = normal_ef_cluster_analysis(
            metadata=meta_test,
            features=Xfeat_test,
            beta=beta_test_a,
            r=r_test_a,
            labels=labels_test
        )

        print(
            "  Normal-EF cluster check | "
            f"n_test={len(meta_test)}, "
            f"n_normal={(meta_test['EF'].values >= EF_CLASSIFICATION_THRESHOLD).sum()}, "
            f"test_clusters={np.unique(labels_test)}, "
            f"normal_clusters={np.unique(labels_test[meta_test['EF'].values >= EF_CLASSIFICATION_THRESHOLD])}",
            flush=True
        )

        if normal_summary is not None and len(normal_summary) > 0:
            normal_summary["repeat"] = repeat_id
            normal_summary["split"] = "test"
            normal_summary["model"] = cluster_name
            normal_summary["k"] = best_k
            normal_cluster_rows.append(normal_summary)

        if normal_comparison is not None and len(normal_comparison) > 0:
            normal_comparison["repeat"] = repeat_id
            normal_comparison["split"] = "test"
            normal_comparison["model"] = cluster_name
            normal_comparison["k"] = best_k
            normal_comparison_rows.append(normal_comparison)

        if repeat_id == 0 and normal_payload is not None:

            plot_normal_ef_cluster_summary_figure(
                beta=normal_payload["beta"],
                r=normal_payload["r"],
                labels=normal_payload["labels"],
                features=normal_payload["features"],
                name="fig_normal_EF_cluster_summary"
            )

    except Exception as e:
        print(f"    [WARN] clustering skipped: {e}")

    clustering_df = (
        pd.concat(clustering_rows, ignore_index=True)
        if len(clustering_rows) > 0
        else pd.DataFrame()
    )

    cluster_summary_df = (
        pd.concat(cluster_summary_rows, ignore_index=True)
        if len(cluster_summary_rows) > 0
        else pd.DataFrame()
    )

    normal_cluster_df = (
        pd.concat(normal_cluster_rows, ignore_index=True)
        if len(normal_cluster_rows) > 0
        else pd.DataFrame()
    )

    normal_comparison_df = (
        pd.concat(normal_comparison_rows, ignore_index=True)
        if len(normal_comparison_rows) > 0
        else pd.DataFrame()
    )

    # --------------------------------------------------------
    # 10. Split table
    # --------------------------------------------------------
    split_df = pd.DataFrame({
        "repeat": repeat_id,
        "patient_id": metadata["patient_id"],
        "split": [
            "train" if i in train_idx else "test"
            for i in range(len(metadata))
        ]
    })

    # --------------------------------------------------------
    # 11. Lambda table
    # --------------------------------------------------------
    lambda_table["repeat"] = repeat_id
    lambda_table["best_lambda"] = best_lambda
    lambda_summary["repeat"] = repeat_id
    lambda_summary["best_lambda"] = best_lambda

    # --------------------------------------------------------
    # 12. Figure payload
    # --------------------------------------------------------
    beta_warped_for_fig = np.concatenate(
        [beta_train_w, beta_test_w],
        axis=0
    )

    r_warped_for_fig = np.concatenate(
        [r_train_w, r_test_w],
        axis=0
    )

    X_warped_for_fig = np.concatenate(
        [X_train_w, X_test_w],
        axis=0
    )

    beta_all_for_fig = np.concatenate(
        [beta_train_a, beta_test_a],
        axis=0
    )

    r_all_for_fig = np.concatenate(
        [r_train_a, r_test_a],
        axis=0
    )

    X_all_for_fig = np.concatenate(
        [X_train_a, X_test_a],
        axis=0
    )

    beta_raw_for_fig = np.concatenate(
        [beta_train, beta_test],
        axis=0
    )

    beta_out_raw_for_fig = np.concatenate(
        [
            beta_out_raw[train_idx],
            beta_out_raw[test_idx]
        ],
        axis=0
    )

    X_raw_for_fig = np.concatenate(
        [X_train, X_test],
        axis=0
    )

    q_raw_for_fig = np.array([
        srvf(X_raw_for_fig[i])
        for i in range(len(X_raw_for_fig))
    ])

    q_hat_for_fig = np.array([
        normalize_srvf(q_raw_for_fig[i])
        for i in range(len(q_raw_for_fig))
    ])

    gammas_for_fig = np.concatenate(
        [gam_train, gam_test],
        axis=0
    )

    save_json({
        'scope': 'inner_CV_training_only_representation_learning',
        'external_train_row_ids': np.asarray(train_idx).tolist(),
        'external_test_row_ids': np.asarray(test_idx).tolist(),
        'groups': [_camus_cv.audit_folds(folds) for folds in inner_feature_cache.values()]
    }, f'inner_cv_audit_repeat_{repeat_id:03d}')

    return {
        "alignment": alignment_df,
        "regression": regression_df,
        "classification": classification_df,
        "clustering": clustering_df,
        "cluster_summary": cluster_summary_df,
        "normal_cluster": normal_cluster_df,
        "normal_cluster_comparison": normal_comparison_df,
        "split": split_df,
        "lambda": lambda_table,
        "lambda_summary": lambda_summary,

        "final_aligned_for_figures": {
            "beta": beta_all_for_fig,
            "r": r_all_for_fig,
            "X": X_all_for_fig
        },

        "pipeline_debug": {
            "beta_in_raw": beta_raw_for_fig,
            "beta_out_raw": beta_out_raw_for_fig,
            "q_raw": q_raw_for_fig,
            "q_hat": q_hat_for_fig,
            "beta_warped": beta_warped_for_fig,
            "r_warped": r_warped_for_fig,
            "X_warped": X_warped_for_fig,
            "beta_aligned": beta_all_for_fig,
            "r_aligned": r_all_for_fig,
            "gammas": gammas_for_fig,
            "X_template": X_template
        }
    }
    
# ============================================================
# 17. Main
# ============================================================

def main():
    print("==========================================", flush=True)
    print("CAMUS OFFSET-SRVF REPEATED TRAIN/TEST WITH TRAIN-CV", flush=True)
    print("WITH REGRESSION, CLASSIFICATION, CLUSTERING", flush=True)
    print("==========================================", flush=True)

    # --------------------------------------------------------
    # 1. Build dataset
    # --------------------------------------------------------
    beta_in_raw, beta_out_raw, r_raw, X_raw, metadata = build_real_dataset()

    metadata.to_csv(DATA_DIR / "metadata.csv", index=False)

    save_clinical_cohort_summary(
        metadata=metadata,
        beta_raw=beta_in_raw,
        r_raw=r_raw
    )

    print(f"Valid patients     : {len(metadata)}")
    print(f"Number of repeats  : {N_REPEATS}")
    print(f"XGBoost available  : {HAS_XGBOOST}")
    print(f"LightGBM available : {HAS_LIGHTGBM}")

    # --------------------------------------------------------
    # 2. Containers
    # --------------------------------------------------------
    all_alignment = []
    all_regression = []
    all_classification = []
    all_clustering = []
    all_cluster_summary = []
    all_normal_cluster = []
    all_normal_cluster_comparison = []
    all_splits = []
    all_lambda = []

    last_beta_aligned = None
    last_r_aligned = None
    pipeline_debug_data = None

    # --------------------------------------------------------
    # 3. Repeated train / validation / test analysis
    # --------------------------------------------------------
    for b in range(N_REPEATS):

        out = run_one_repeat(
            repeat_id=b,
            beta_in_raw=beta_in_raw,
            beta_out_raw=beta_out_raw,
            r_raw=r_raw,
            X_raw=X_raw,
            metadata=metadata
        )

        all_alignment.append(out["alignment"])

        if len(out["regression"]) > 0:
            all_regression.append(out["regression"])

        if len(out["classification"]) > 0:
            all_classification.append(out["classification"])

        if len(out["clustering"]) > 0:
            all_clustering.append(out["clustering"])

        if len(out["cluster_summary"]) > 0:
            all_cluster_summary.append(out["cluster_summary"])

        if len(out["normal_cluster"]) > 0:
            all_normal_cluster.append(out["normal_cluster"])

        if len(out["normal_cluster_comparison"]) > 0:
            all_normal_cluster_comparison.append(
                out["normal_cluster_comparison"]
            )

        all_splits.append(out["split"])
        all_lambda.append(out["lambda"])

        last_beta_aligned = out["final_aligned_for_figures"]["beta"]
        last_r_aligned = out["final_aligned_for_figures"]["r"]

        if b == 0:
            pipeline_debug_data = out["pipeline_debug"]

    # --------------------------------------------------------
    # 4. Combine repeated outputs
    # --------------------------------------------------------
    alignment_results = pd.concat(all_alignment, ignore_index=True)

    regression_results = (
        pd.concat(all_regression, ignore_index=True)
        if len(all_regression) > 0
        else pd.DataFrame()
    )

    classification_results = (
        pd.concat(all_classification, ignore_index=True)
        if len(all_classification) > 0
        else pd.DataFrame()
    )

    clustering_results = (
        pd.concat(all_clustering, ignore_index=True)
        if len(all_clustering) > 0
        else pd.DataFrame()
    )

    cluster_summary_results = (
        pd.concat(all_cluster_summary, ignore_index=True)
        if len(all_cluster_summary) > 0
        else pd.DataFrame()
    )

    normal_cluster_results = (
        pd.concat(all_normal_cluster, ignore_index=True)
        if len(all_normal_cluster) > 0
        else pd.DataFrame()
    )

    normal_cluster_comparison_results = (
        pd.concat(all_normal_cluster_comparison, ignore_index=True)
        if len(all_normal_cluster_comparison) > 0
        else pd.DataFrame()
    )

    split_results = pd.concat(all_splits, ignore_index=True)
    lambda_results = pd.concat(all_lambda, ignore_index=True)

    # --------------------------------------------------------
    # 5. Save raw repeated tables
    # --------------------------------------------------------
    alignment_results.to_csv(
        TAB_DIR / "alignment_repeated_results.csv",
        index=False
    )

    regression_results.to_csv(
        TAB_DIR / "regression_repeated_results.csv",
        index=False
    )

    classification_results.to_csv(
        TAB_DIR / "classification_repeated_results.csv",
        index=False
    )

    clustering_results.to_csv(
        TAB_DIR / "clustering_repeated_results.csv",
        index=False
    )

    cluster_summary_results.to_csv(
        TAB_DIR / "cluster_summary_repeated_results.csv",
        index=False
    )

    normal_cluster_results.to_csv(
        TAB_DIR / "normal_EF_cluster_repeated_results.csv",
        index=False
    )

    normal_cluster_comparison_results.to_csv(
        TAB_DIR / "normal_EF_cluster_comparison_repeated_results.csv",
        index=False
    )

    split_results.to_csv(
        DATA_DIR / "repeated_splits.csv",
        index=False
    )

    lambda_results.to_csv(
        TAB_DIR / "lambda_repeated_results.csv",
        index=False
    )

    # --------------------------------------------------------
    # 6. Alignment summary
    # --------------------------------------------------------
    alignment_summary = summarize_results(
        alignment_results,
        group_cols=["split"],
        metric_cols=[
            "srvf_geodesic_before_mean",
            "srvf_geodesic_before_sd",
            "srvf_geodesic_after_mean",
            "srvf_geodesic_after_sd",
            "srvf_geodesic_change_percent"
        ]
    )

    save_latex_table(
        alignment_summary,
        "alignment_repeated_summary",
        "Repeated train/test alignment performance measured by SRVF Hilbert-sphere geodesic distance.",
        "tab:alignment_repeated_summary"
    )

    # --------------------------------------------------------
    # 7. Regression summary
    # --------------------------------------------------------
    if len(regression_results) > 0:

        regression_summary = summarize_results(
            regression_results,
            group_cols=["outcome", "model"],
            metric_cols=["mae", "rmse", "r2", "corr"]
        )

        save_latex_table(
            regression_summary,
            "regression_repeated_summary",
            "Repeated test-set regression performance. Hyperparameters were selected by cross-validation within the training set.",
            "tab:regression_repeated_summary"
        )

        best_regression = (
            regression_summary
            .sort_values(["outcome", "rmse_mean"])
            .groupby("outcome")
            .head(1)
            .reset_index(drop=True)
        )

        save_latex_table(
            best_regression,
            "regression_best_models",
            "Best regression model for each outcome based on mean test RMSE across repetitions.",
            "tab:regression_best_models"
        )

    # --------------------------------------------------------
    # 8. Classification summary
    # --------------------------------------------------------
    if len(classification_results) > 0:

        classification_summary = summarize_results(
            classification_results,
            group_cols=["outcome", "model"],
            metric_cols=[
                "accuracy",
                "balanced_accuracy",
                "f1",
                "auc",
                "sensitivity",
                "specificity"
            ]
        )

        save_latex_table(
            classification_summary,
            "classification_repeated_summary",
            "Repeated test-set EF classification performance. Hyperparameters were selected by cross-validation within the training set.",
            "tab:classification_repeated_summary"
        )

        best_classification = (
            classification_summary
            .sort_values(["outcome", "auc_mean"], ascending=[True, False])
            .groupby("outcome")
            .head(1)
            .reset_index(drop=True)
        )

        save_latex_table(
            best_classification,
            "classification_best_models",
            "Best classification model based on mean test AUC across repetitions.",
            "tab:classification_best_models"
        )

    # --------------------------------------------------------
    # 9. Lambda summary
    # --------------------------------------------------------
    lambda_frequency = (
        lambda_results[["repeat", "best_lambda"]]
        .drop_duplicates()
        .groupby("best_lambda")
        .size()
        .reset_index(name="frequency")
    )

    save_lambda_selection_stability(lambda_results)

    save_latex_table(
        lambda_frequency,
        "lambda_selection_frequency",
        "Frequency of selected lambda values across repeated training-set cross-validation g splits.",
        "tab:lambda_selection_frequency"
    )

    plot_lambda_summary(lambda_results)
    plot_lambda_curve(lambda_results)

    # --------------------------------------------------------
    # 10. Clustering summaries
    # --------------------------------------------------------
    if len(clustering_results) > 0:

        save_clustering_model_selection_table(clustering_results)

        selected_cluster_frequency = (
            clustering_results[["repeat", "selected_model", "selected_k"]]
            .drop_duplicates()
            .groupby(["selected_model", "selected_k"])
            .size()
            .reset_index(name="frequency")
        )

        save_latex_table(
            selected_cluster_frequency,
            "clustering_selected_model_k_frequency",
            "Frequency of selected clustering model and number of clusters across repeated splits.",
            "tab:clustering_selected_model_k_frequency"
        )

    if len(cluster_summary_results) > 0:

        cluster_numeric_cols = [
            c for c in cluster_summary_results.columns
            if c not in ["repeat", "split", "model", "cluster"]
            and pd.api.types.is_numeric_dtype(cluster_summary_results[c])
        ]

        if len(cluster_numeric_cols) > 0:
            cluster_summary_mean = summarize_results(
                cluster_summary_results,
                group_cols=["split", "cluster"],
                metric_cols=cluster_numeric_cols
            )

            save_latex_table(
                cluster_summary_mean,
                "cluster_summary_repeated",
                "Repeated cluster-level clinical summaries across train and test splits.",
                "tab:cluster_summary_repeated"
            )


    # --------------------------------------------------------
    # Best model hyperparameter tables
    # --------------------------------------------------------
    save_best_model_hyperparameter_tables(
        regression_results=regression_results,
        classification_results=classification_results
    )

    # --------------------------------------------------------
    # 11. Normal EF phenotype discovery summaries
    # --------------------------------------------------------
    if len(normal_cluster_results) > 0:

        normal_numeric_cols = [
            c for c in normal_cluster_results.columns
            if c not in ["repeat", "split", "model", "cluster"]
            and pd.api.types.is_numeric_dtype(normal_cluster_results[c])
        ]

        if len(normal_numeric_cols) > 0:
            normal_cluster_summary = summarize_results(
                normal_cluster_results,
                group_cols=["cluster"],
                metric_cols=normal_numeric_cols
            )

            save_latex_table(
                normal_cluster_summary,
                "normal_EF_cluster_summary",
                "Repeated structural cluster summaries among patients with normal EF.",
                "tab:normal_EF_cluster_summary"
            )

    if len(normal_cluster_comparison_results) > 0:

        comparison_numeric_cols = [
            c for c in normal_cluster_comparison_results.columns
            if c not in ["repeat", "split", "model", "metric"]
            and pd.api.types.is_numeric_dtype(normal_cluster_comparison_results[c])
        ]

        if len(comparison_numeric_cols) > 0:
            normal_comparison_summary = summarize_results(
                normal_cluster_comparison_results,
                group_cols=["metric"],
                metric_cols=comparison_numeric_cols
            )

            save_latex_table(
                normal_comparison_summary,
                "normal_EF_cluster_comparison",
                "Repeated comparison of structural clusters among patients with normal EF.",
                "tab:normal_EF_cluster_comparison"
            )

    save_simplified_normal_ef_cluster_table(
        normal_cluster_comparison_results
    )
    
    # --------------------------------------------------------
    # 12. Main publication figures
    # --------------------------------------------------------
    if last_beta_aligned is not None and last_r_aligned is not None:

        plot_population_mean_contours(
            last_beta_aligned,
            last_r_aligned,
            pipeline_debug_data["X_template"]
        )

        plot_thickness_modes(
            last_r_aligned
        )

    if pipeline_debug_data is not None:

        plot_clean_real_data_pipeline(
            beta_in_raw=pipeline_debug_data["beta_in_raw"],
            beta_out_raw=pipeline_debug_data["beta_out_raw"],
            q_raw=pipeline_debug_data["q_raw"],
            q_hat=pipeline_debug_data["q_hat"],
            beta_warped=pipeline_debug_data["beta_warped"],
            beta_aligned=pipeline_debug_data["beta_aligned"],
            r_aligned=pipeline_debug_data["r_aligned"],
            gammas=pipeline_debug_data["gammas"],
            X_template=pipeline_debug_data["X_template"]
        )

    # --------------------------------------------------------
    # 13. Save run information
    # --------------------------------------------------------
    run_info = {
        "valid_patients": int(len(metadata)),
        "n_repeats": int(N_REPEATS),
        "train_size": float(1.0 - TEST_SIZE),
        "test_size": float(TEST_SIZE),
        "random_state": int(RANDOM_STATE),
        "alpha_embed": float(ALPHA_EMBED),
        "n_grid_points": int(N),
        "lambda_grid": LAMBDA_GRID,
        "xgboost_available": bool(HAS_XGBOOST),
        "lightgbm_available": bool(HAS_LIGHTGBM),
        "outputs": {
            "figures": str(FIG_DIR),
            "tables": str(TAB_DIR),
            "data": str(DATA_DIR),
            "tex": str(TEX_DIR)
        }
    }

    save_json(run_info, "run_info")

    with open(REAL_OUT / "README_outputs.txt", "w") as f:
        f.write(
            "CAMUS Offset-SRVF Repeated Train/Test Outputs with Training-Set CV\n"
            "=================================================================\n\n"
            "Methodological rule:\n"
            "  All tuning is performed by cross-validation within the training set.\n"
            "  The test set is used only once for final evaluation.\n\n"
            "Analyses:\n"
            "  1. Embedded SRVF alignment\n"
            "  2. Lambda selection by K-fold CV on the training set\n"
            "  3. Alignment quality measured by SRVF Hilbert-sphere geodesic distance\n"
            "  4. Regression with training-set cross-validated hyperparameter tuning\n"
            "  5. Classification with training-set cross-validated hyperparameter tuning\n"
            "  6. Clustering and normal-EF phenotype discovery\n\n"
            "Folders:\n"
            "  figures/ : PNG, PDF, EPS figures\n"
            "  tables/  : CSV and LaTeX tables\n"
            "  data/    : metadata, repeated splits, JSON summaries\n"
            "  tex/     : LaTeX figure snippets\n\n"
            f"Number of repetitions: {N_REPEATS}\n"
            f"Train/test split: {1.0 - TEST_SIZE:.2f}/{TEST_SIZE:.2f}\n"
            f"Training CV folds: {N_FOLDS}\n"
            f"Embedding alpha: {ALPHA_EMBED}\n"
            f"Grid points: {N}\n"
            f"XGBoost available: {HAS_XGBOOST}\n"
            f"LightGBM available: {HAS_LIGHTGBM}\n"
        )

    print("==========================================")
    print("DONE")
    print("Outputs saved in:", REAL_OUT)
    print("==========================================")

print("ABOUT TO CALL MAIN", flush=True)

if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Real-data lambda CV curve from saved tables/
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# ------------------------------------------------------------
# folders
# ------------------------------------------------------------
OUT_DIR = Path("runs/camus")

TAB_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# read table
# ------------------------------------------------------------
df = pd.read_csv(
    TAB_DIR / "lambda_repeated_results.csv"
)

# ------------------------------------------------------------
# aggregate
# ------------------------------------------------------------
summary = (
    df.groupby("lambda")
      .agg(
          mean_cv=("srvf_geodesic_validation_mse", "mean"),
          sd_cv=("srvf_geodesic_validation_mse", "std")
      )
      .reset_index()
      .sort_values("lambda")
)

lambdas = summary["lambda"].values
mean_scores = summary["mean_cv"].values
sd_scores = summary["sd_cv"].fillna(0).values

# ------------------------------------------------------------
# best lambda
# ------------------------------------------------------------
best_idx = np.argmin(mean_scores)

best_lambda = lambdas[best_idx]
best_score = mean_scores[best_idx]

print(f"Selected lambda = {best_lambda}")
print(f"Minimum CV error = {best_score:.6f}")

# ------------------------------------------------------------
# figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(
    lambdas,
    mean_scores,
    marker="o",
    linewidth=4,
    markersize=11,
    label="Mean 5-fold CV SRVF geodesic MSE"
)

ax.fill_between(
    lambdas,
    mean_scores - sd_scores,
    mean_scores + sd_scores,
    alpha=0.20,
    label=r"$\pm1$ SD"
)

# ------------------------------------------------------------
# selected lambda
# ------------------------------------------------------------
ax.axvline(
    best_lambda,
    color="red",
    linestyle="--",
    linewidth=2.5,
    alpha=0.9,
    label=rf"Selected $\lambda^\ast={best_lambda:.3g}$"
)

ax.scatter(
    best_lambda,
    best_score,
    color="red",
    s=260,
    marker="*",
    edgecolor="black",
    linewidth=1.2,
    zorder=20
)

ax.annotate(
    rf"$\lambda^\ast={best_lambda:.3g}$",
    xy=(best_lambda, best_score),
    xytext=(15, 15),
    textcoords="offset points",
    fontsize=16,
    color="red",
    weight="bold"
)

# ------------------------------------------------------------
# formatting
# ------------------------------------------------------------
ax.set_xscale("log")

ax.set_xlabel(
    r"$\lambda$",
    fontsize=22
)

ax.set_ylabel(
    "SRVF geodesic validation MSE",
    fontsize=22
)

ax.set_title(
    "5-fold CV lambda selection curve",
    fontsize=26,
    pad=18
)

ax.tick_params(
    axis="both",
    labelsize=18
)

ax.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.35
)

ax.legend(
    fontsize=15,
    frameon=True,
    loc="upper left"
)

plt.tight_layout()

# ------------------------------------------------------------
# save
# ------------------------------------------------------------
fig.savefig(
    FIG_DIR / "fig_lambda_curve_real_data.pdf",
    bbox_inches="tight"
)

fig.savefig(
    FIG_DIR / "fig_lambda_curve_real_data.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(FIG_DIR / "fig_lambda_curve_real_data.pdf")
print(FIG_DIR / "fig_lambda_curve_real_data.png")